# **PQC Benchmark Analysis**

### Empirical evaluation of NIST-standardised post-quantum cryptography on ARM64 IoT hardware

**Author:** Hugo Hernández Moreno  
**Institution:** MSMK University — Pearson BTEC HND  
**Hardware:** Raspberry Pi 5 (BCM2712, ARM Cortex-A76, 8 GB RAM)  
**Dataset:** 16,000 measurements — 1,000 iterations × 16 operations × 5 algorithms

---

### **Research question**

> Are NIST-standardised post-quantum cryptography algorithms (ML-KEM/Kyber-512 and ML-DSA/ML-DSA-44) superior to traditional algorithms (RSA-2048 and ECC-256) for IoT devices based on modern ARM64 architecture, considering operational efficiency — latency, energy consumption, memory usage — and differential resistance against the quantum threat projected for 2028–2035?

---

### **Notebook structure**

| Section | Content |
|---------|---------|
| S1 Setup | Imports, colour palette, theme system, data loading |
| S2 EDA | Data quality, warm-up, outliers, distributions, correlations |
| S3 Descriptive | Summary statistics, within-family distributions, CV, skewness |
| S4 Inferential | Normality, Kruskal-Wallis, Mann-Whitney U, effect sizes, Bonferroni |
| S5 Shor | Quantum threat simulation — RSA-15, RSA-21, extrapolation to RSA-2048 |
| S6 Summary | Master comparison table, radar chart, TLS deployment scenarios |

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy.stats import shapiro, kruskal, mannwhitneyu, spearmanr
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

In [2]:
# Paths
PROJECT_ROOT = Path.cwd().parent
DATA_PATH    = PROJECT_ROOT / 'data' / 'results'
FIG_DARK     = PROJECT_ROOT / 'figures' / 'dark'
FIG_LIGHT    = PROJECT_ROOT / 'figures' / 'light'

for p in [FIG_DARK, FIG_LIGHT]:
    p.mkdir(parents=True, exist_ok=True)

In [3]:
# Algorithm color palette
COLORS = {
    'rsa'   : '#ff5757',   # red    — legacy, threat
    'kyber' : '#ff914d',   # orange — PQC KEM
    'mldsa' : '#c084fc',   # violet — PQC signature
    'ecdh'  : '#38bdf8',   # sky    — classical ECC KEM
    'ecdsa' : '#34d399',   # emerald — classical ECC signature
}

# Convenience: map full dataset name → color key
def alg_color(name: str) -> str:
    name = name.lower()
    for key in COLORS:
        if key in name:
            return COLORS[key]
    return '#888888'

In [4]:
# Theme definitions
_THEMES = {
    'dark': dict(
        bg          = '#191919',
        panel       = '#1e1e1e',
        grid        = '#2a2a2a',
        text        = '#ffffff',
        subtext     = '#aaaaaa',
        axis        = '#555555',
        annotation_bg = 'rgba(30,30,30,0.92)',
        annotation_border = '#555555',
        paper_bgcolor = '#191919',
        plot_bgcolor  = '#1e1e1e',
        font_color    = '#ffffff',
    ),
    'light': dict(
        bg          = '#ffffff',
        panel       = '#f8f8f8',
        grid        = '#e5e5e5',
        text        = '#1a1a1a',
        subtext     = '#666666',
        axis        = '#cccccc',
        annotation_bg = 'rgba(255,255,255,0.95)',
        annotation_border = '#cccccc',
        paper_bgcolor = '#ffffff',
        plot_bgcolor  = '#f8f8f8',
        font_color    = '#1a1a1a',
    ),
}

In [5]:
# Active theme
_ACTIVE_MODE = 'dark'   # 'dark' | 'light'
T = _THEMES[_ACTIVE_MODE]

In [6]:
def apply_theme(fig: go.Figure, mode: str = None) -> go.Figure:
    m = _THEMES[mode or _ACTIVE_MODE]

    fig.update_layout(
        paper_bgcolor = m['paper_bgcolor'],
        plot_bgcolor  = m['plot_bgcolor'],
        font          = dict(family='Arial', color=m['font_color'], size=12),
        title_font    = dict(family='Arial', color=m['font_color'], size=16),
        legend        = dict(
            bgcolor      = m['annotation_bg'],
            bordercolor  = m['annotation_border'],
            borderwidth  = 1,
            font         = dict(color=m['font_color'], size=11),
        ),
        hoverlabel    = dict(
            bgcolor    = m['annotation_bg'],
            bordercolor= m['annotation_border'],
            font_color = m['font_color'],
            font_size  = 11,
        ),
    )

    fig.update_xaxes(
        gridcolor     = m['grid'],
        zerolinecolor = m['axis'],
        linecolor     = m['axis'],
        tickfont      = dict(color=m['subtext'], size=11),
        title_font    = dict(color=m['text'], size=12),
    )
    fig.update_yaxes(
        gridcolor     = m['grid'],
        zerolinecolor = m['axis'],
        linecolor     = m['axis'],
        tickfont      = dict(color=m['subtext'], size=11),
        title_font    = dict(color=m['text'], size=12),
    )

    return fig

In [124]:
def save_fig(fig: go.Figure, filename: str, modes: list = None):
    if modes is None:
        modes = ['dark', 'light']

    for mode in modes:
        fig_copy = go.Figure(fig)
        apply_theme(fig_copy, mode)
        out_dir = FIG_DARK if mode == 'dark' else FIG_LIGHT
        fig_copy.write_html(out_dir / f'{filename}.html')

        # Uncomment when kaleido is installed for static export:
        # fig_copy.write_image(out_dir / f'{filename}.png', scale=2, width=1200, height=600)

    print(f"  Saved: figures/dark/{filename}.html")
    print(f"  Saved: figures/light/{filename}.html")


print("✓ Theme system loaded")
print(f"  Active mode : {_ACTIVE_MODE}")
print(f"  Algorithms  : {list(COLORS.keys())}")


✓ Theme system loaded
  Active mode : dark
  Algorithms  : ['rsa', 'kyber', 'mldsa', 'ecdh', 'ecdsa']


---

## **S1 — Setup**

Imports, global colour palette, theme system, and data loading.  
The `COLORS` dict and `apply_theme()` function are used by every figure in this notebook — define once, apply everywhere.  
Change `_ACTIVE_MODE` to `'light'` to regenerate all figures in light mode for the thesis PDF.

In [ ]:
# Upload datasets
from pathlib import Path
import pandas as pd


# Relative paths
PROJECT_ROOT = Path(__file__).parent.parent if '__file__' in dir() else Path.cwd().parent
DATA_PATH    = PROJECT_ROOT / 'data' / 'results'
FIGURES_PATH = PROJECT_ROOT / 'figures'

print(f"Data exists  : {DATA_PATH.exists()}")

# Upload CSVs
df_collection = {}
for csv_file in DATA_PATH.glob('*.csv'):
    df_collection[csv_file.stem] = pd.read_csv(csv_file)

print(f"\n{len(df_collection)} files loaded")
print(f"Total records: {sum(len(df) for df in df_collection.values()):,}")

# Categorization of datasets
category = {
    'kyber': {
        'keygen': df_collection.get('kyber512_keygen'),
        'encaps': df_collection.get('kyber512_encaps'),
        'decaps': df_collection.get('kyber512_decaps')
    },
    'mldsa': {
        'keygen': df_collection.get('mldsa44_keygen'),
        'sign':   df_collection.get('mldsa44_sign'),
        'verify': df_collection.get('mldsa44_verify')
    },
    'rsa': {
        'keygen':   df_collection.get('rsa2048_keygen'),
        'encrypt':  df_collection.get('rsa2048_encrypt'),
        'decrypt':  df_collection.get('rsa2048_decrypt'),
        'sign':     df_collection.get('rsa2048_sign'),
        'verify':   df_collection.get('rsa2048_verify')
    },
    'ecdh': {
        'keygen': df_collection.get('ecdh256_keygen'),
        'derive': df_collection.get('ecdh256_derive')
    },
    'ecdsa': {
        'keygen': df_collection.get('ecdsa256_keygen'),
        'sign':   df_collection.get('ecdsa256_sign'),
        'verify': df_collection.get('ecdsa256_verify')
    }
}

Data exists  : True

16 files loaded
Total records: 16,000


---

## **S2 — Exploratory Data Analysis (EDA)**


Before any statistical inference, we validate the integrity of the collected data and explore its distributional properties. This section answers five questions:

1. **Is the dataset complete?** All 16 operations must have exactly 1,000 observations with no nulls or negative latencies.
2. **Is there a warm-up effect?** If the first iterations are systematically slower (JIT, cache cold-start), they would bias the distribution.
3. **Are outliers explainable?** High-latency outliers are expected in OS-scheduled benchmarks (context switches, GC pauses). We identify and document them.
4. **What is the full distributional shape?** Median and IQR alone do not reveal skewness, bimodality, or heavy tails. Violin plots expose the complete density.
5. **Is latency correlated with thermal output?** If faster algorithms also run cooler, temperature is a redundant metric.

In [31]:
# Quality check table (text output)

print("Data quality check:\n")
print(f"{'Operation':<30} {'Rows':>6}  {'Nulls':>6}  {'Neg latencies':>14}")
print("-" * 60)
for name, df in sorted(df_collection.items()):
    nulls    = int(df.isnull().sum().sum())
    neg      = int((df['time_ms'] < 0).sum())
    status   = "✓" if (len(df) == 1000 and nulls == 0 and neg == 0) else "✗"
    print(f"{status} {name:<28} {len(df):>6}  {nulls:>6}  {neg:>14}")

Data quality check:

Operation                        Rows   Nulls   Neg latencies
------------------------------------------------------------
✓ ecdh256_derive                 1000       0               0
✓ ecdh256_keygen                 1000       0               0
✓ ecdsa256_keygen                1000       0               0
✓ ecdsa256_sign                  1000       0               0
✓ ecdsa256_verify                1000       0               0
✓ kyber512_decaps                1000       0               0
✓ kyber512_encaps                1000       0               0
✓ kyber512_keygen                1000       0               0
✓ mldsa44_keygen                 1000       0               0
✓ mldsa44_sign                   1000       0               0
✓ mldsa44_verify                 1000       0               0
✓ rsa2048_decrypt                1000       0               0
✓ rsa2048_encrypt                1000       0               0
✓ rsa2048_keygen                 1000       0     

In [125]:
def plot_eda_completeness(df_collection: dict) -> go.Figure:

    names      = sorted(df_collection.keys())
    row_counts = [len(df_collection[n]) for n in names]
    medians    = [float(df_collection[n]['time_ms'].median()) for n in names]
    mins       = [float(df_collection[n]['time_ms'].min())    for n in names]
    maxs       = [float(df_collection[n]['time_ms'].max())    for n in names]
    colors_bar = ['#34d399' if r == 1000 else '#ff5757' for r in row_counts]

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            '<b>Dataset completeness</b>',
            '<b>Latency range per operation</b>',
        ),
        column_widths=[0.38, 0.62],   # right panel gets more space
        horizontal_spacing=0.08,
    )

    # Left — row counts
    fig.add_trace(go.Bar(
        x=row_counts, y=names,
        orientation='h',
        marker_color=colors_bar,
        marker_line_color=colors_bar,
        marker_line_width=1,
        opacity=0.85,
        text=[str(r) for r in row_counts],
        textposition='outside',
        textfont=dict(size=10, color=T['text']),
        showlegend=False,
        hovertemplate='<b>%{y}</b><br>Rows: %{x}<extra></extra>',
    ), row=1, col=1)

    fig.add_vline(
        x=1000,
        line=dict(color=T['subtext'], width=1.5, dash='dash'),
        row=1, col=1,
    )

    # Right — min/max range + median
    for i, name in enumerate(names):
        color = alg_color(name)
        fig.add_trace(go.Scatter(
            x=[mins[i], maxs[i]], y=[name, name],
            mode='lines',
            line=dict(color=color, width=2),
            opacity=0.45,
            showlegend=False,
            hoverinfo='skip',
        ), row=1, col=2)
        fig.add_trace(go.Scatter(
            x=[medians[i]], y=[name],
            mode='markers',
            marker=dict(color=color, size=9, line=dict(color='white', width=1.2)),
            showlegend=False,
            hovertemplate=f'<b>{name}</b><br>Median: {medians[i]:.4f} ms<extra></extra>',
        ), row=1, col=2)

    fig.update_xaxes(
        type='log',
        title_text='Latency (ms) — log scale',
        showgrid=True,
        row=1, col=2,
    )
    fig.update_xaxes(
        title_text='Row count',
        range=[0, 1150],   # leave room for "100" labels
        showgrid=True,
        row=1, col=1,
    )
    fig.update_yaxes(
        tickfont=dict(size=10),
        row=1, col=1,
    )
    fig.update_yaxes(
        showticklabels=False,  # names already visible on left panel
        row=1, col=2,
    )

    fig.update_layout(
        height=560,
        width=1300,          # wider overall canvas
        margin=dict(t=80, b=60, l=160, r=60),  # less left margin
    )
    apply_theme(fig)
    return fig


fig_completeness = plot_eda_completeness(df_collection)
fig_completeness.show()
save_fig(fig_completeness, 'eda_completeness')

  Saved: figures/dark/eda_completeness.html
  Saved: figures/light/eda_completeness.html


In [126]:
# Warm-up validation — first 10 vs remaining 990
print("Warm-up validation:\n")
print(f"{'Operation':<30} {'First 10 (ms)':>14}  {'Rest 990 (ms)':>14}  {'Diff (%)':>9}  {'Status':>8}")
print("-" * 82)
for name, df in sorted(df_collection.items()):
    first_10 = df.head(10)['time_ms'].mean()
    rest_990 = df.tail(990)['time_ms'].mean()
    diff_pct = abs(first_10 - rest_990) / rest_990 * 100
    if diff_pct > 20:
        status = "⚠ WARNING"
    elif diff_pct > 10:
        status = "~ CAUTION"
    else:
        status = "✓ OK"
    print(f"  {name:<28} {first_10:>14.4f}  {rest_990:>14.4f}  {diff_pct:>8.1f}%  {status:>9}")

Warm-up validation:

Operation                       First 10 (ms)   Rest 990 (ms)   Diff (%)    Status
----------------------------------------------------------------------------------
  ecdh256_derive                       0.2688          0.2497       7.6%       ✓ OK
  ecdh256_keygen                       0.1206          0.1196       0.8%       ✓ OK
  ecdsa256_keygen                      0.1271          0.1190       6.7%       ✓ OK
  ecdsa256_sign                        0.1862          0.1881       1.0%       ✓ OK
  ecdsa256_verify                      0.3718          0.3596       3.4%       ✓ OK
  kyber512_decaps                      0.0727          0.0714       1.9%       ✓ OK
  kyber512_encaps                      0.1012          0.0983       2.9%       ✓ OK
  kyber512_keygen                      0.0700          0.0822      14.8%  ~ CAUTION
  mldsa44_keygen                       0.2292          0.2128       7.7%       ✓ OK
  mldsa44_sign                         1.0028          0.

In [ ]:
def plot_warmup(df_collection: dict) -> go.Figure:

    rows = []
    for name, df in sorted(df_collection.items()):
        first_10 = float(df.head(10)['time_ms'].mean())
        rest_990 = float(df.tail(990)['time_ms'].mean())
        diff_pct = abs(first_10 - rest_990) / rest_990 * 100
        rows.append({
            'operation': name,
            'first_10' : first_10,
            'rest_990' : rest_990,
            'diff_pct' : diff_pct,
        })

    df_wu = pd.DataFrame(rows).sort_values('diff_pct')

    def wu_color(pct):
        if pct > 20:  return '#ff5757'   # red — warm-up artifact
        elif pct > 10: return '#ff914d'  # orange — caution
        else:          return '#34d399'  # green — ok

    colors = [wu_color(p) for p in df_wu['diff_pct']]

    fig = go.Figure(go.Bar(
        x            = df_wu['diff_pct'],
        y            = df_wu['operation'],
        orientation  = 'h',
        marker_color = colors,
        marker_line_color = colors,
        marker_line_width = 1,
        opacity      = 0.85,
        text         = [f'{p:.1f}%' for p in df_wu['diff_pct']],
        textposition = 'outside',
        textfont     = dict(size=10, color=T['text']),
        hovertemplate=(
            '<b>%{y}</b><br>'
            'Difference: %{x:.2f}%<extra></extra>'
        ),
    ))

    # Threshold reference lines
    fig.add_vline(
        x=10,
        line=dict(color='#ff914d', width=1.5, dash='dash'),
        annotation_text='Caution (10%)',
        annotation_font_color='#ff914d',
        annotation_font_size=10,
        annotation_position='top right',
    )
    fig.add_vline(
        x=20,
        line=dict(color='#ff5757', width=1.5, dash='dash'),
        annotation_text='Warm-up artifact (20%)',
        annotation_font_color='#ff5757',
        annotation_font_size=10,
        annotation_position='top right',
    )

    fig.update_layout(
        title=dict(
            text=(
                '<b>Warm-up validation — first 10 vs remaining 990 iterations</b><br>'
                '<sup>Threshold 20%: conservative for ARM Cortex-A76 cache cold-start</sup>'
            ),
            x=0.5,
        ),
        xaxis=dict(title='Mean latency difference (%)', showgrid=True),
        yaxis=dict(title=''),
        height=560,
        width=900,
        margin=dict(t=100, b=60, l=200, r=80),
        showlegend=False,
    )

    apply_theme(fig)
    return fig


fig_warmup = plot_warmup(df_collection)
fig_warmup.show()
save_fig(fig_warmup, 'eda_warmup')

  Saved: figures/dark/eda_warmup.html
  Saved: figures/light/eda_warmup.html


In [34]:
# Print table
print("Outlier detection — Tukey extended fence (Q3 + 3×IQR):\n")
print(f"{'Operation':<30} {'Outliers':>8}  {'Rate (%)':>9}  {'Fence (ms)':>12}")
print("-" * 65)
for name, df in sorted(df_collection.items()):
    q1    = df['time_ms'].quantile(0.25)
    q3    = df['time_ms'].quantile(0.75)
    fence = q3 + 3*(q3-q1)
    n_out = int((df['time_ms'] > fence).sum())
    pct   = n_out / len(df) * 100
    print(f"  {name:<28} {n_out:>8}  {pct:>8.1f}%  {fence:>12.4f}")

Outlier detection — Tukey extended fence (Q3 + 3×IQR):

Operation                      Outliers   Rate (%)    Fence (ms)
-----------------------------------------------------------------
  ecdh256_derive                    202      20.2%        0.2512
  ecdh256_keygen                     20       2.0%        0.1331
  ecdsa256_keygen                    22       2.2%        0.1322
  ecdsa256_sign                      51       5.1%        0.2045
  ecdsa256_verify                   192      19.2%        0.3679
  kyber512_decaps                    35       3.5%        0.0786
  kyber512_encaps                    41       4.1%        0.1078
  kyber512_keygen                     3       0.3%        0.1296
  mldsa44_keygen                      9       0.9%        0.2494
  mldsa44_sign                       12       1.2%        2.4971
  mldsa44_verify                    150      15.0%        0.2252
  rsa2048_decrypt                    69       6.9%        3.6483
  rsa2048_encrypt                

In [35]:
def plot_outliers(df_collection: dict) -> go.Figure:
    """
    Two-panel:
      Left:  outlier percentage per operation (bar)
      Right: outlier absolute count with threshold annotation
    """
    rows = []
    for name, df in sorted(df_collection.items()):
        q1  = float(df['time_ms'].quantile(0.25))
        q3  = float(df['time_ms'].quantile(0.75))
        iqr = q3 - q1
        fence = q3 + 3 * iqr
        n_out = int((df['time_ms'] > fence).sum())
        rows.append({
            'operation'   : name,
            'outlier_pct' : n_out / len(df) * 100,
            'outlier_n'   : n_out,
            'threshold_ms': fence,
            'alg'         : name.split('_')[0],
        })

    df_out = pd.DataFrame(rows).sort_values('outlier_pct')

    def out_color(pct):
        if pct > 15:  return '#ff5757'
        elif pct > 5: return '#ff914d'
        else:         return '#34d399'

    colors = [out_color(p) for p in df_out['outlier_pct']]

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            '<b>Outlier rate (%)</b>',
            '<b>Outlier count (absolute)</b>',
        ),
        horizontal_spacing=0.12,
    )

    # Left — percentage
    fig.add_trace(go.Bar(
        x=df_out['outlier_pct'], y=df_out['operation'],
        orientation='h',
        marker_color=colors, marker_line_color=colors, marker_line_width=1,
        opacity=0.85,
        text=[f'{p:.1f}%' for p in df_out['outlier_pct']],
        textposition='outside',
        textfont=dict(size=10, color=T['text']),
        showlegend=False,
        hovertemplate='<b>%{y}</b><br>Outlier rate: %{x:.2f}%<extra></extra>',
    ), row=1, col=1)

    fig.add_vline(x=5,  line=dict(color='#ff914d', width=1, dash='dash'), row=1, col=1)
    fig.add_vline(x=15, line=dict(color='#ff5757', width=1, dash='dash'), row=1, col=1)

    # Right — absolute count
    fig.add_trace(go.Bar(
        x=df_out['outlier_n'], y=df_out['operation'],
        orientation='h',
        marker_color=[alg_color(n) for n in df_out['operation']],
        opacity=0.8,
        text=[str(n) for n in df_out['outlier_n']],
        textposition='outside',
        textfont=dict(size=10, color=T['text']),
        showlegend=False,
        hovertemplate=(
            '<b>%{y}</b><br>'
            'Outliers: %{x}<br>'
            'Threshold: %{customdata:.4f} ms<extra></extra>'
        ),
        customdata=df_out['threshold_ms'],
    ), row=1, col=2)

    fig.update_xaxes(title_text='Outlier rate (%)', row=1, col=1)
    fig.update_xaxes(title_text='Count (out of 1,000)', row=1, col=2)

    fig.update_layout(
        height=520, width=1100,
        margin=dict(t=80, b=60, l=200, r=60),
    )
    apply_theme(fig)
    return fig


fig_outliers = plot_outliers(df_collection)
fig_outliers.show()
save_fig(fig_outliers, 'eda_outliers')

  Saved: figures/dark/eda_outliers.html
  Saved: figures/light/eda_outliers.html


In [128]:
def plot_violins_by_family(df_collection: dict) -> go.Figure:
    """
    5-panel violin plot, one per algorithm family.
    Independent Y scale per panel — essential for readability
    given the 2,800× latency range across families.
    """
    families = [
        ('Kyber-512', COLORS['kyber'],
         ['kyber512_keygen', 'kyber512_encaps', 'kyber512_decaps']),
        ('ML-DSA-44', COLORS['mldsa'],
         ['mldsa44_keygen', 'mldsa44_sign', 'mldsa44_verify']),
        ('RSA-2048',  COLORS['rsa'],
         ['rsa2048_keygen', 'rsa2048_encrypt', 'rsa2048_decrypt',
          'rsa2048_sign', 'rsa2048_verify']),
        ('ECDH-256',  COLORS['ecdh'],
         ['ecdh256_keygen', 'ecdh256_derive']),
        ('ECDSA-256', COLORS['ecdsa'],
         ['ecdsa256_keygen', 'ecdsa256_sign', 'ecdsa256_verify']),
    ]

    fig = make_subplots(
        rows=1, cols=5,
        subplot_titles=[f'<b>{f[0]}</b>' for f in families],
        horizontal_spacing=0.05,
    )

    for col, (label, color, ops) in enumerate(families, start=1):
        for op_name in ops:
            if op_name not in df_collection:
                continue
            short = op_name.split('_')[-1].capitalize()   # keygen→Keygen, sign→Sign etc.

            fig.add_trace(go.Violin(
                y                = df_collection[op_name]['time_ms'],
                name             = short,
                legendgroup      = label,
                showlegend       = False,
                line_color       = color,
                fillcolor        = color,
                opacity          = 0.65,
                box_visible      = True,
                meanline_visible = True,
                meanline_color   = 'white',
                box_line_color   = 'white',
                points           = False,
                width            = 0.8,
                hovertemplate    = (
                    f'<b>{label} — {short}</b><br>'
                    'Value: %{y:.4f} ms<extra></extra>'
                ),
            ), row=1, col=col)

        short_ops = [o.split('_')[-1].capitalize() for o in ops
                     if o in df_collection]
        fig.update_xaxes(
            tickvals  = list(range(len(short_ops))),
            ticktext  = short_ops,
            tickfont  = dict(size=10),
            row=1, col=col,
        )
        fig.update_yaxes(
            title_text = 'Latency (ms)' if col == 1 else '',
            showgrid   = True,
            row=1, col=col,
        )

    fig.update_layout(
        title=dict(
            text=(
                '<b>Latency distribution — all operations by family</b><br>'
                '<sup>Independent Y scale per family | '
                'Inner box = IQR | White line = mean | n=1,000</sup>'
            ),
            x=0.5,
        ),
        height = 520,
        width  = 1300,
        margin = dict(t=100, b=60, l=70, r=40),
        showlegend = False,
    )

    apply_theme(fig)
    return fig


fig_violins = plot_violins_by_family(df_collection)
fig_violins.show()
save_fig(fig_violins, 'eda_violin_all_operations')

  Saved: figures/dark/eda_violin_all_operations.html
  Saved: figures/light/eda_violin_all_operations.html


In [138]:
def plot_latency_temp_correlation(df_collection: dict) -> go.Figure:

    rows = []
    for name, df in df_collection.items():
        # Short label: "rsa2048_sign" → "RSA Sign"
        parts = name.split('_')
        family_raw = parts[0]
        op_raw     = '_'.join(parts[1:])
        family_map = {
            'kyber512': ('kyber', 'Kyber'),
            'mldsa44' : ('mldsa', 'MLDSA'),
            'rsa2048' : ('rsa',   'RSA'),
            'ecdh256' : ('ecdh',  'ECDH'),
            'ecdsa256': ('ecdsa', 'ECDSA'),
        }
        alg_key, alg_short = family_map.get(family_raw, (family_raw, family_raw))
        rows.append({
            'operation'  : name,
            'label'      : f'{alg_short} {op_raw.capitalize()}',
            'latency_med': float(df['time_ms'].median()),
            'temp_mean'  : float(df['temp_delta_c'].mean()),
            'alg_key'    : alg_key,
        })

    df_corr = pd.DataFrame(rows)
    r_stat, p_val = spearmanr(df_corr['latency_med'], df_corr['temp_mean'])

    # Label positions — offset colliding points manually
    label_positions = {
        'kyber512_keygen' : 'top right',
        'kyber512_encaps' : 'bottom right',
        'kyber512_decaps' : 'top left',
        'mldsa44_keygen'  : 'top right',
        'mldsa44_sign'    : 'bottom right',
        'mldsa44_verify'  : 'top right',
        'rsa2048_keygen'  : 'top left',
        'rsa2048_encrypt' : 'bottom right',
        'rsa2048_decrypt' : 'top right',
        'rsa2048_sign'    : 'bottom left',
        'rsa2048_verify'  : 'top right',
        'ecdh256_keygen'  : 'bottom left',
        'ecdh256_derive'  : 'top right',
        'ecdsa256_keygen' : 'top right',
        'ecdsa256_sign'   : 'bottom right',
        'ecdsa256_verify' : 'bottom right',
    }

    families = {
        'kyber': 'Kyber-512',
        'mldsa': 'ML-DSA-44',
        'rsa'  : 'RSA-2048',
        'ecdh' : 'ECDH-256',
        'ecdsa': 'ECDSA-256',
    }

    fig = go.Figure()

    for key, label in families.items():
        sub = df_corr[df_corr['alg_key'] == key]
        if sub.empty:
            continue

        positions = [label_positions.get(row['operation'], 'top center')
                     for _, row in sub.iterrows()]

        fig.add_trace(go.Scatter(
            x    = sub['latency_med'],
            y    = sub['temp_mean'],
            mode = 'markers+text',
            name = label,
            marker = dict(
                color = COLORS[key],
                size  = 15,
                line  = dict(color='white', width=1.5),
            ),
            text         = sub['label'],
            textposition = positions,
            textfont     = dict(size=9, color=T['subtext']),
            hovertemplate=(
                '<b>%{text}</b><br>'
                'Median latency: %{x:.4f} ms<br>'
                'Mean ΔT: %{y:.4f}°C<extra></extra>'
            ),
        ))

    # Zero thermal line
    fig.add_hline(
        y=0,
        line=dict(color=T['axis'], width=1.5, dash='dot'),
        annotation_text='Thermal neutral (ΔT = 0)',
        annotation_font_color=T['subtext'],
        annotation_font_size=10,
        annotation_position='bottom right',
    )

    # Spearman result annotation
    sig_text = 'p<0.05 — significant' if p_val < 0.05 else f'p={p_val:.3f} — not significant'
    interp = ('Weak: temperature is an <b>independent</b> metric' if abs(r_stat) < 0.3
              else 'Moderate: partial overlap between metrics' if abs(r_stat) < 0.6
              else 'Strong: temperature closely tracks latency')

    fig.add_annotation(
        xref='paper', yref='paper',
        x=0.65, y=0.02,
        text=(
            f'<b>Spearman ρ = {r_stat:.3f}</b><br>'
            f'{sig_text}<br>'
            f'→ {interp}'
        ),
        showarrow=False,
        align='left',
        xanchor='left',
        yanchor='bottom',
        font=dict(size=11, color=T['text']),
        bgcolor=T['annotation_bg'],
        bordercolor=T['annotation_border'],
        borderwidth=1,
        borderpad=8,
    )

    fig.update_layout(
        title=dict(
            text=(
                '<b>Latency vs thermal output — are they correlated?</b><br>'
                '<sup>Each point = one operation | Spearman ρ (non-parametric) | '
                'n=1,000 per operation</sup>'
            ),
            x=0.5,
        ),
        xaxis=dict(
            title='Median latency (ms) — log scale',
            type='log',
            showgrid=True,
        ),
        yaxis=dict(
            title='Mean temperature delta (°C)',
            showgrid=True,
        ),
        legend=dict(
            title=dict(text='Algorithm family', font=dict(size=11)),
            orientation='v',
            x=1.01, y=1.0,
            xanchor='left',
        ),
        height=580,
        width=1000,
        margin=dict(t=100, b=80, l=80, r=160),
    )

    apply_theme(fig)
    return fig, r_stat, p_val


fig_corr, r_stat, p_val = plot_latency_temp_correlation(df_collection)
fig_corr.show()
save_fig(fig_corr, 'eda_latency_temp_correlation')

print(f"\nSpearman ρ = {r_stat:.4f}  |  p = {p_val:.4f}")
if abs(r_stat) < 0.3:
    print("→ Weak correlation: temperature is an INDEPENDENT metric.")
    print("  Kyber's thermal profile is structurally different from RSA,")
    print("  not merely a consequence of being faster.")
elif abs(r_stat) < 0.6:
    print("→ Moderate correlation: partial overlap between metrics.")
else:
    print("→ Strong correlation: temperature closely tracks latency.")

  Saved: figures/dark/eda_latency_temp_correlation.html
  Saved: figures/light/eda_latency_temp_correlation.html

Spearman ρ = 0.0824  |  p = 0.7617
→ Weak correlation: temperature is an INDEPENDENT metric.
  Kyber's thermal profile is structurally different from RSA,
  not merely a consequence of being faster.


### **S2.5 — Thermal throttle validation**

Having established the cumulative thermal impact, we confirm that no operation caused thermal throttling during the benchmark. The RPi 5 (BCM2712) throttles the CPU at ~85°C by reducing clock frequency — if this occurred, latency measurements would be inflated and invalid.  
This figure validates the integrity of all latency data in S3 and S4.

In [105]:
def plot_thermal_throttle_validation(categories: dict) -> go.Figure:
    THROTTLE_THRESHOLD = 85.0   # °C — RPi 5 BCM2712 hard limit
    SAFE_THRESHOLD     = 60.0   # °C — conservative safe zone

    rows = []
    for alg, ops in categories.items():
        for op, df in ops.items():
            if df is None:
                continue
            rows.append({
                'operation': f'{alg.upper()} {op.capitalize()}',
                'alg_key'  : alg,
                'temp_max' : float(df['temp_absolute_c'].max()),
                'temp_mean': float(df['temp_absolute_c'].mean()),
            })

    df_t = pd.DataFrame(rows).sort_values('temp_max', ascending=False)

    colors = [COLORS[row['alg_key']] for _, row in df_t.iterrows()]

    fig = go.Figure()

    # Max temperature bars
    fig.add_trace(go.Bar(
        x            = df_t['operation'],
        y            = df_t['temp_max'],
        name         = 'Max temperature',
        marker_color = colors,
        marker_line_color = colors,
        marker_line_width = 1,
        opacity      = 0.85,
        text         = [f'{v:.1f}°C' for v in df_t['temp_max']],
        textposition = 'outside',
        textfont     = dict(size=9, color=T['text']),
        hovertemplate=(
            '<b>%{x}</b><br>'
            'Max temp: %{y:.1f}°C<extra></extra>'
        ),
    ))

    # Mean temperature overlay (scatter dots)
    fig.add_trace(go.Scatter(
        x    = df_t['operation'],
        y    = df_t['temp_mean'],
        mode = 'markers',
        name = 'Mean temperature',
        marker=dict(
            color  = 'white',
            size   = 7,
            symbol = 'diamond',
            line   = dict(color=T['subtext'], width=1.5),
        ),
        hovertemplate=(
            '<b>%{x}</b><br>'
            'Mean temp: %{y:.1f}°C<extra></extra>'
        ),
    ))

    # Throttling threshold
    fig.add_hline(
        y=THROTTLE_THRESHOLD,
        line=dict(color=COLORS['rsa'], width=2, dash='dash'),
        annotation_text=f'Throttling threshold ({THROTTLE_THRESHOLD}°C)',
        annotation_font_color=COLORS['rsa'],
        annotation_font_size=10,
        annotation_position='top right',
    )

    # Safe zone threshold
    fig.add_hline(
        y=SAFE_THRESHOLD,
        line=dict(color='#ff914d', width=1.5, dash='dot'),
        annotation_text=f'Safe zone (<{SAFE_THRESHOLD}°C)',
        annotation_font_color='#ff914d',
        annotation_font_size=10,
        annotation_position='bottom right',
    )

    # Summary annotation
    temp_max_global = float(df_t['temp_max'].max())
    margin = THROTTLE_THRESHOLD - temp_max_global

    fig.add_annotation(
        xref='paper', yref='paper',
        x=0.01, y=0.98,
        text=(
            f'<b>Throttling validation</b><br>'
            f'Global max: {temp_max_global:.1f}°C<br>'
            f'Margin to throttle: {margin:.1f}°C<br>'
            f'Result: <b>✓ No throttling detected</b>'
        ),
        showarrow=False,
        align='left',
        font=dict(size=11, color=T['text']),
        bgcolor=T['annotation_bg'],
        bordercolor='#34d399',
        borderwidth=1.5,
        borderpad=8,
        xanchor='left',
        yanchor='top',
    )

    fig.update_layout(
        title=dict(
            text=(
                '<b>Absolute maximum temperature per operation</b><br>'
                '<sup>Validates absence of thermal throttling during benchmark | '
                'RPi 5 BCM2712 throttle threshold: 85°C</sup>'
            ),
            x=0.5,
        ),
        xaxis=dict(
            tickangle = -40,
            tickfont  = dict(size=9),
            showgrid  = False,
        ),
        yaxis=dict(
            title   = 'Temperature (°C)',
            showgrid= True,
            range   = [0, THROTTLE_THRESHOLD + 8],
        ),
        legend=dict(orientation='h', y=-0.22, x=0.5, xanchor='center'),
        height  = 560,
        width   = 1100,
        margin  = dict(t=110, b=110, l=70, r=60),
        barmode = 'group',
    )

    apply_theme(fig)
    return fig


fig_throttle = plot_thermal_throttle_validation(category)
fig_throttle.show()
save_fig(fig_throttle, 'thermal_throttle_validation')


  Saved: figures/dark/thermal_throttle_validation.html
  Saved: figures/light/thermal_throttle_validation.html


In [106]:
# Print summary
print("\nThermal throttling validation:")
print(f"  Throttle threshold : 85.0°C")
for _, row in pd.DataFrame([{
    'operation': f'{alg.upper()} {op}',
    'temp_max' : float(category[alg][op]['temp_absolute_c'].max())
} for alg, ops in category.items()
  for op, df in ops.items() if df is not None
]).sort_values('temp_max', ascending=False).head(3).iterrows():
    print(f"  Hottest: {row['operation']:<28} {row['temp_max']:.1f}°C")


Thermal throttling validation:
  Throttle threshold : 85.0°C
  Hottest: RSA encrypt                  40.0°C
  Hottest: RSA sign                     40.0°C
  Hottest: RSA decrypt                  39.5°C


### **S2.6 — Sample Dataset Inspection**

Raw view of one dataset (Kyber-512 KeyGen) to confirm column structure, data types, and value ranges before proceeding to statistical analysis.

In [140]:
example_dataset= category['kyber']['keygen']

print("KYBER-512 KEYGEN - DATASET")

pd.set_option('display.max_rows', 20)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

display(example_dataset.head(10))

# Restaurar configuración por defecto
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.width')
pd.reset_option('display.max_colwidth')

KYBER-512 KEYGEN - DATASET


,iteration,timestamp,time_ms,memory_process_mb,memory_system_mb,temp_delta_c,temp_absolute_c,current_ma
0,1,2025-12-05T15:30:28.246712,0.063149,0.015625,0.0,0.0,32.3,0.2
1,2,2025-12-05T15:30:28.252332,0.072223,0.000000,0.0,0.0,31.8,0.0
2,3,2025-12-05T15:30:28.257843,0.069556,0.000000,0.0,0.0,32.9,0.2
3,4,2025-12-05T15:30:28.263356,0.071093,0.000000,0.0,0.0,32.3,-0.4
4,5,2025-12-05T15:30:28.268835,0.070111,0.000000,0.0,0.0,32.9,-0.1
5,6,2025-12-05T15:30:28.274281,0.070796,0.000000,0.0,0.0,32.9,-0.1
6,7,2025-12-05T15:30:28.279774,0.070593,0.015625,0.0,0.0,34.0,0.1
7,8,2025-12-05T15:30:28.285297,0.070834,0.000000,0.0,0.0,32.3,0.5
8,9,2025-12-05T15:30:28.290758,0.071444,0.000000,0.0,0.0,32.3,-0.4
9,10,2025-12-05T15:30:28.296268,0.070685,0.000000,0.0,0.0,32.3,0.0


---

## **S3 — Descriptive Statistics**

Descriptive statistics characterise the central tendency, spread, and shape of each operation's latency distribution.

| Metric | Interpretation |
|--------|---------------|
| **Median** | Robust central tendency — unaffected by outliers. Preferred over mean for skewed distributions. |
| **IQR** | Spread of the central 50% of observations. Robust measure of variability. |
| **CV (%)** | Coefficient of variation — relative variability normalised by mean. Enables cross-algorithm consistency comparison. |
| **Skewness γ₁** | Asymmetry of the distribution. Positive skew (right tail) is structurally expected in benchmarks. |
| **P95 / P99** | Worst-case latency — critical for real-time IoT systems with latency budgets. |

**Why median over mean?** All distributions are non-normal (confirmed in §4). The mean is pulled toward the right tail by outliers. The median is the statistically correct central tendency estimator for skewed data.

In [39]:
from scipy.stats import skew as scipy_skew

def build_stats_table(category: dict) -> pd.DataFrame:
    rows = []
    for alg, ops in category.items():
        for op, df in ops.items():
            if df is None:
                continue
            t = df['time_ms']
            q1  = float(t.quantile(0.25))
            q3  = float(t.quantile(0.75))
            rows.append({
                'Algorithm' : f'{alg.upper()} {op.capitalize()}',
                'Family'    : alg,
                'n'         : len(t),
                'Median'    : float(t.median()),
                'Mean'      : float(t.mean()),
                'Std'       : float(t.std()),
                'IQR'       : q3 - q1,
                'CV (%)'    : float(t.std() / t.mean() * 100),
                'Skewness'  : float(scipy_skew(t)),
                'Min'       : float(t.min()),
                'Max'       : float(t.max()),
                'P95'       : float(t.quantile(0.95)),
                'P99'       : float(t.quantile(0.99)),
            })

    return pd.DataFrame(rows).sort_values('Median').reset_index(drop=True)


df_stats = build_stats_table(category)

In [141]:
# ── Display with conditional formatting
fmt = {
    'Median'  : '{:.6f}',
    'Mean'    : '{:.6f}',
    'Std'     : '{:.6f}',
    'IQR'     : '{:.6f}',
    'CV (%)'  : '{:.2f}',
    'Skewness': '{:.3f}',
    'Min'     : '{:.6f}',
    'Max'     : '{:.6f}',
    'P95'     : '{:.6f}',
    'P99'     : '{:.6f}',
}

display(
    df_stats.drop(columns=['Family'])
    .style
    .format(fmt)
    .background_gradient(subset=['Median'], cmap='RdYlGn_r')
    .background_gradient(subset=['CV (%)'],  cmap='YlOrRd')
    .background_gradient(subset=['Skewness'], cmap='PuOr_r')
    .hide(axis='index')
    .set_caption('Latency descriptive statistics — all 16 operations (n=1,000 each)')
)

print(f"\nFastest operation : {df_stats.iloc[0]['Algorithm']} "
    f"({df_stats.iloc[0]['Median']:.6f} ms)")
print(f"Slowest operation : {df_stats.iloc[-1]['Algorithm']} "
    f"({df_stats.iloc[-1]['Median']:.6f} ms)")
print(f"Speed ratio       : {df_stats.iloc[-1]['Median']/df_stats.iloc[0]['Median']:.0f}×")

Algorithm,n,Median,Mean,Std,IQR,CV (%),Skewness,Min,Max,P95,P99
KYBER Decaps,1000,0.070000,0.071395,0.006654,0.002361,9.32,8.482,0.064963,0.151482,0.077853,0.084997
KYBER Keygen,1000,0.084686,0.082123,0.009681,0.014501,11.79,1.597,0.063149,0.169723,0.094372,0.110022
KYBER Encaps,1000,0.096556,0.098373,0.007247,0.003111,7.37,5.770,0.083315,0.183242,0.107243,0.126488
ECDSA Keygen,1000,0.117251,0.119124,0.006621,0.004070,5.56,5.155,0.109612,0.201242,0.129410,0.141417
ECDH Keygen,1000,0.117871,0.119655,0.006753,0.004212,5.64,4.645,0.108352,0.186982,0.130245,0.138594
ECDSA Sign,1000,0.185241,0.188119,0.008558,0.005213,4.55,2.374,0.173612,0.256205,0.204538,0.210817
MLDSA Keygen,1000,0.208779,0.212970,0.011481,0.010709,5.39,2.273,0.195057,0.289335,0.235095,0.244168
MLDSA Verify,1000,0.210983,0.215709,0.011995,0.003897,5.56,2.809,0.205927,0.303224,0.238950,0.265560
RSA Encrypt,1000,0.223677,0.208220,0.029948,0.053759,14.38,-0.109,0.157371,0.326131,0.250817,0.258588
RSA Verify,1000,0.230159,0.228985,0.020910,0.004389,9.13,-1.168,0.161242,0.326557,0.257021,0.269638



Fastest operation : KYBER Decaps (0.070000 ms)
Slowest operation : RSA Keygen (198.606352 ms)
Speed ratio       : 2837×


In [142]:
def plot_family_violins(category: dict) -> go.Figure:
    families = [
        ('kyber', 'Kyber-512',  ['keygen', 'encaps', 'decaps']),
        ('mldsa', 'ML-DSA-44',  ['keygen', 'sign', 'verify']),
        ('rsa',   'RSA-2048',   ['keygen', 'encrypt', 'decrypt', 'sign', 'verify']),
        ('ecdh',  'ECDH-256',   ['keygen', 'derive']),
        ('ecdsa', 'ECDSA-256',  ['keygen', 'sign', 'verify']),
    ]

    fig = make_subplots(
        rows=1, cols=5,
        subplot_titles=[f'<b>{f[1]}</b>' for f in families],
        horizontal_spacing=0.04,
    )

    for col, (key, label, ops) in enumerate(families, start=1):
        color = COLORS[key]
        for op in ops:
            df = category[key].get(op)
            if df is None:
                continue
            fig.add_trace(go.Violin(
                y                = df['time_ms'],
                name             = op.capitalize(),
                legendgroup      = label,
                showlegend       = False,
                line_color       = color,
                fillcolor        = color,
                opacity          = 0.65,
                box_visible      = True,
                meanline_visible = True,
                meanline_color   = 'white',
                box_line_color   = 'white',
                points           = False,
                width            = 0.85,
                hovertemplate    = (
                    f'<b>{label} — {op}</b><br>'
                    'Value: %{y:.4f} ms<extra></extra>'
                ),
            ), row=1, col=col)

        # Operation labels on X axis
        fig.update_xaxes(
            tickvals=list(range(len(ops))),
            ticktext=[o.capitalize() for o in ops],
            tickfont=dict(size=9),
            row=1, col=col,
        )
        # Independent Y scale per family (crucial for readability)
        fig.update_yaxes(
            title_text='Latency (ms)' if col == 1 else '',
            showgrid=True,
            row=1, col=col,
        )

    fig.update_layout(
        title=dict(
            text=(
                '<b>Within-family latency distributions — all operations</b><br>'
                '<sup>Independent Y scale per family | Inner box = IQR | White line = mean</sup>'
            ),
            x=0.5,
        ),
        height=520,
        width=1200,
        margin=dict(t=100, b=60, l=70, r=40),
        showlegend=False,
    )

    apply_theme(fig)
    return fig


fig_family_violins = plot_family_violins(category)
fig_family_violins.show()
save_fig(fig_family_violins, 'desc_violin_by_family')

  Saved: figures/dark/desc_violin_by_family.html
  Saved: figures/light/desc_violin_by_family.html


In [45]:
def plot_cv_skewness(df_stats: pd.DataFrame) -> go.Figure:
    df = df_stats.copy().sort_values('CV (%)')

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            '<b>Coefficient of Variation (%)</b><br>'
            '<sup>Lower = more consistent</sup>',
            '<b>Skewness (γ₁)</b><br>'
            '<sup>Higher = heavier right tail</sup>',
        ),
        horizontal_spacing=0.12,
    )

    colors_cv = [alg_color(row['Algorithm']) for _, row in df.iterrows()]

    # Left — CV
    fig.add_trace(go.Bar(
        x            = df['CV (%)'],
        y            = df['Algorithm'],
        orientation  = 'h',
        marker_color = colors_cv,
        marker_line_color = colors_cv,
        marker_line_width = 1,
        opacity      = 0.82,
        text         = [f'{v:.1f}%' for v in df['CV (%)']],
        textposition = 'outside',
        textfont     = dict(size=9, color=T['text']),
        showlegend   = False,
        hovertemplate='<b>%{y}</b><br>CV: %{x:.2f}%<extra></extra>',
    ), row=1, col=1)

    # Right — Skewness (sorted by skewness)
    df_sk = df_stats.sort_values('Skewness')
    colors_sk = [alg_color(row['Algorithm']) for _, row in df_sk.iterrows()]

    fig.add_trace(go.Bar(
        x            = df_sk['Skewness'],
        y            = df_sk['Algorithm'],
        orientation  = 'h',
        marker_color = colors_sk,
        marker_line_color = colors_sk,
        marker_line_width = 1,
        opacity      = 0.82,
        text         = [f'{v:.2f}' for v in df_sk['Skewness']],
        textposition = 'outside',
        textfont     = dict(size=9, color=T['text']),
        showlegend   = False,
        hovertemplate='<b>%{y}</b><br>Skewness: %{x:.3f}<extra></extra>',
    ), row=1, col=2)

    # Zero line for skewness
    fig.add_vline(
        x=0,
        line=dict(color=T['axis'], width=1, dash='dot'),
        row=1, col=2,
    )

    fig.update_xaxes(title_text='CV (%)', showgrid=True, row=1, col=1)
    fig.update_xaxes(title_text='Skewness γ₁', showgrid=True, row=1, col=2)

    fig.update_layout(
        height=560,
        width=1100,
        margin=dict(t=100, b=60, l=200, r=80),
    )

    apply_theme(fig)
    return fig


fig_cv_skew = plot_cv_skewness(df_stats)
fig_cv_skew.show()
save_fig(fig_cv_skew, 'desc_cv_skewness')

# Narrative summary
print("\nConsistency ranking (CV — lower is better):")
top3  = df_stats.nsmallest(3, 'CV (%)')
bot3  = df_stats.nlargest(3, 'CV (%)')
for _, row in top3.iterrows():
    print(f"  ✓ {row['Algorithm']:<28} CV = {row['CV (%)']:.2f}%")
print("  ...")
for _, row in bot3.iterrows():
    print(f"  ✗ {row['Algorithm']:<28} CV = {row['CV (%)']:.2f}%")

print("\nSkewness (right tail weight — higher = more extreme outliers):")
for _, row in df_stats.nlargest(3, 'Skewness').iterrows():
    print(f"  {row['Algorithm']:<28} γ₁ = {row['Skewness']:.3f}")

  Saved: figures/dark/desc_cv_skewness.html
  Saved: figures/light/desc_cv_skewness.html

Consistency ranking (CV — lower is better):
  ✓ ECDSA Sign                   CV = 4.55%
  ✓ ECDH Derive                  CV = 4.89%
  ✓ ECDSA Verify                 CV = 4.91%
  ...
  ✗ RSA Keygen                   CV = 62.90%
  ✗ MLDSA Sign                   CV = 60.32%
  ✗ RSA Encrypt                  CV = 14.38%

Skewness (right tail weight — higher = more extreme outliers):
  KYBER Decaps                 γ₁ = 8.482
  KYBER Encaps                 γ₁ = 5.770
  ECDSA Keygen                 γ₁ = 5.155


### **S3.2 — Latency comparison by operation type**

Box plots comparing the three key operation families: KeyGen, Sign, and the derived Speedup bar chart. Each box shows the full IQR, median (line), and mean (diamond marker). Log scale is used for KeyGen due to the RSA-2048 range difference (~2,000× spread).

In [46]:
def plot_keygen_boxplot(category: dict) -> go.Figure:

    algs = [
        ('kyber', 'keygen', 'Kyber-512'),
        ('rsa',   'keygen', 'RSA-2048'),
        ('ecdh',  'keygen', 'ECDH-256'),
    ]

    fig = go.Figure()

    medianas = []
    for key, op, label in algs:
        data = category[key][op]['time_ms']
        medianas.append(float(data.median()))

        fig.add_trace(go.Box(
            y          = data,
            name       = label,
            marker_color = COLORS[key],
            line_color   = COLORS[key],
            fillcolor    = COLORS[key],
            opacity      = 0.75,
            boxmean      = True,      # shows mean as dashed line
            boxpoints    = False,     # no individual points — cleaner
            hovertemplate = (
                f'<b>{label}</b><br>'
                'Median: %{median:.4f} ms<br>'
                'Q1: %{q1:.4f} ms<br>'
                'Q3: %{q3:.4f} ms<extra></extra>'
            ),
        ))

    # Speedup annotations
    speedup_kr = medianas[1] / medianas[0]
    speedup_er = medianas[1] / medianas[2]

    fig.add_annotation(
        xref='paper', yref='paper',
        x=0.02, y=0.98,
        text=(
            f'<b>Speedups vs RSA-2048</b><br>'
            f'Kyber-512: <b>{speedup_kr:.0f}×</b> faster<br>'
            f'ECDH-256:  <b>{speedup_er:.0f}×</b> faster'
        ),
        showarrow=False,
        align='left',
        font=dict(size=11, color=T['text']),
        bgcolor=T['annotation_bg'],
        bordercolor=T['annotation_border'],
        borderwidth=1,
        borderpad=8,
    )

    fig.update_layout(
        title=dict(
            text=(
                '<b>KeyGen latency — Kyber-512 vs RSA-2048 vs ECDH-256</b><br>'
                '<sup>n=1,000 iterations per algorithm | Raspberry Pi 5 ARM64</sup>'
            ),
            x=0.5,
        ),
        yaxis=dict(
            title='Latency (ms) — log scale',
            type='log',
            showgrid=True,
        ),
        xaxis=dict(title='Algorithm'),
        showlegend=False,
        height=560,
        width=900,
        margin=dict(t=100, b=60, l=80, r=40),
    )

    apply_theme(fig)
    return fig


fig_keygen = plot_keygen_boxplot(category)
fig_keygen.show()
save_fig(fig_keygen, 'keygen_boxplot')

# Print summary
medianas_print = {
    k: float(category[k]['keygen']['time_ms'].median())
    for k in ['kyber', 'rsa', 'ecdh']
}
print(f"\nMedians:")
for k, v in medianas_print.items():
    print(f"  {k:<8}: {v:.4f} ms")
print(f"\nSpeedups vs RSA:")
for k in ['kyber', 'ecdh']:
    print(f"  {k:<8}: {medianas_print['rsa']/medianas_print[k]:.0f}×")

  Saved: figures/dark/keygen_boxplot.html
  Saved: figures/light/keygen_boxplot.html

Medians:
  kyber   : 0.0847 ms
  rsa     : 198.6064 ms
  ecdh    : 0.1179 ms

Speedups vs RSA:
  kyber   : 2345×
  ecdh    : 1685×


In [ ]:
def plot_sign_boxplot(category: dict) -> go.Figure:

    algs = [
        ('mldsa', 'sign', 'ML-DSA-44'),
        ('rsa',   'sign', 'RSA-2048'),
        ('ecdsa', 'sign', 'ECDSA-256'),
    ]

    fig = go.Figure()

    medianas = []
    for key, op, label in algs:
        data = category[key][op]['time_ms']
        medianas.append(float(data.median()))

        fig.add_trace(go.Box(
            y            = data,
            name         = label,
            marker_color = COLORS[key],
            line_color   = COLORS[key],
            fillcolor    = COLORS[key],
            opacity      = 0.75,
            boxmean      = True,
            boxpoints    = False,
            hovertemplate=(
                f'<b>{label}</b><br>'
                'Median: %{median:.4f} ms<br>'
                'Q1: %{q1:.4f} ms<br>'
                'Q3: %{q3:.4f} ms<extra></extra>'
            ),
        ))

    speedup_mr = medianas[1] / medianas[0]   # RSA vs ML-DSA
    speedup_er = medianas[1] / medianas[2]   # RSA vs ECDSA

    fig.add_annotation(
        xref='paper', yref='paper',
        x=0.02, y=0.98,
        text=(
            f'<b>Speedups vs RSA-2048</b><br>'
            f'ML-DSA-44: <b>{speedup_mr:.1f}×</b> faster<br>'
            f'ECDSA-256: <b>{speedup_er:.1f}×</b> faster'
        ),
        showarrow=False,
        align='left',
        font=dict(size=11, color=T['text']),
        bgcolor=T['annotation_bg'],
        bordercolor=T['annotation_border'],
        borderwidth=1,
        borderpad=8,
    )

    fig.update_layout(
        title=dict(
            text=(
                '<b>Sign latency — ML-DSA-44 vs RSA-2048 vs ECDSA-256</b><br>'
                '<sup>n=1,000 iterations per algorithm | Raspberry Pi 5 ARM64</sup>'
            ),
            x=0.5,
        ),
        yaxis=dict(title='Latency (ms)', showgrid=True),
        xaxis=dict(title='Algorithm'),
        showlegend=False,
        height=560,
        width=900,
        margin=dict(t=100, b=60, l=80, r=40),
    )

    apply_theme(fig)
    return fig


fig_sign = plot_sign_boxplot(category)
fig_sign.show()
save_fig(fig_sign, 'sign_boxplot')

  Saved: figures/dark/sign_boxplot.html
  Saved: figures/light/sign_boxplot.html


In [ ]:
def plot_speedup_bar(category: dict) -> go.Figure:
    """
    Horizontal bar chart: speedup of each algorithm vs RSA-2048
    baseline. Color = algorithm family. Annotated values.
    """
    # Medians
    med = {
        alg: {op: float(category[alg][op]['time_ms'].median())
              for op in ops if category[alg].get(op) is not None}
        for alg, ops in [
            ('kyber', ['keygen']),
            ('mldsa', ['sign']),
            ('rsa',   ['keygen', 'sign']),
            ('ecdh',  ['keygen']),
            ('ecdsa', ['sign']),
        ]
    }

    comparisons = [
        ('Kyber-512 vs RSA — KeyGen', med['rsa']['keygen'] / med['kyber']['keygen'], 'kyber'),
        ('ML-DSA-44 vs RSA — Sign',   med['rsa']['sign']   / med['mldsa']['sign'],   'mldsa'),
        ('ECDH-256 vs RSA — KeyGen',  med['rsa']['keygen'] / med['ecdh']['keygen'],  'ecdh'),
        ('ECDSA-256 vs RSA — Sign',   med['rsa']['sign']   / med['ecdsa']['sign'],   'ecdsa'),
    ]

    labels   = [c[0] for c in comparisons]
    speedups = [c[1] for c in comparisons]
    colors   = [COLORS[c[2]] for c in comparisons]

    fig = go.Figure(go.Bar(
        x            = speedups,
        y            = labels,
        orientation  = 'h',
        marker_color = colors,
        marker_line_color = [c for c in colors],
        marker_line_width = 1.5,
        opacity      = 0.85,
        text         = [f'{s:.0f}×' for s in speedups],
        textposition = 'outside',
        textfont     = dict(size=12, color=T['text']),
        hovertemplate='<b>%{y}</b><br>Speedup: %{x:.1f}×<extra></extra>',
    ))

    # Reference line at 1× (RSA baseline)
    fig.add_vline(
        x=1,
        line=dict(color=COLORS['rsa'], width=1.5, dash='dot'),
        annotation_text='RSA baseline',
        annotation_font_color=COLORS['rsa'],
        annotation_font_size=10,
        annotation_position='top right',
    )

    max_speedup = max(speedups)
    fig.update_layout(
        title=dict(
            text=(
                '<b>Speedups: PQC and ECC vs RSA-2048</b><br>'
                '<sup>Based on medians | n=1,000 iterations</sup>'
            ),
            x=0.5,
        ),
        xaxis=dict(
            title='Speedup (× faster than RSA-2048)',
            range=[0, max_speedup * 1.20],
            showgrid=True,

        ),
        yaxis=dict(title='', autorange='reversed'),
        height=420,
        width=900,
        margin=dict(t=100, b=60, l=240, r=80),
        showlegend=False,
    )

    apply_theme(fig)
    return fig


fig_speedup = plot_speedup_bar(category)
fig_speedup.show()
save_fig(fig_speedup, 'speedup_bar')

  Saved: figures/dark/speedup_bar.html
  Saved: figures/light/speedup_bar.html


### **S3.3 — Thermal analysis**

Cumulative temperature delta over 1,000 iterations per operation. Negative values indicate net cooling — the RPi 5 thermal management system dissipates heat faster than the algorithm generates it. This is the case for Kyber-512 KeyGen, which is mechanistically significant: lattice polynomial multiplication is computationally cheap enough to leave thermal headroom.

In [114]:
# Build thermal summary (used by plot and print)
thermal_rows = []
for alg, ops in category.items():
    for op, df in ops.items():
        if df is None:
            continue
        thermal_rows.append({
            'Algorithm'         : f'{alg.upper()} {op.capitalize()}',
            'alg_key'           : alg,
            'Median latency (ms)': float(df['time_ms'].median()),
            'Mean ΔT (°C)'      : float(df['temp_delta_c'].mean()),
            'Cumulative ΔT (°C)': float(df['temp_delta_c'].sum()),
        })

df_thermal = pd.DataFrame(thermal_rows).sort_values('Cumulative ΔT (°C)')

In [115]:
# ── Plotly bar chart
fig_thermal = plot_thermal(category)   # defined in §1 setup
fig_thermal.show()
save_fig(fig_thermal, 'thermal_cumulative')

  Saved: figures/dark/thermal_cumulative.html
  Saved: figures/light/thermal_cumulative.html


In [ ]:
# ── Console summary
print("Thermal summary — 1,000 iterations\n")
print(f"  {'Operation':<28} {'Mean ΔT':>9}  {'Cumul. ΔT':>11}")
print(f"  {'-'*52}")
for _, row in df_thermal.iterrows():
    print(f"  {row['Algorithm']:<28} "
          f"{row['Mean ΔT (°C)']:>+8.3f}°C  "
          f"{row['Cumulative ΔT (°C)']:>+10.2f}°C")

Thermal summary — 1,000 iterations

  Operation                      Mean ΔT    Cumul. ΔT
  ----------------------------------------------------
  MLDSA Keygen                   -0.014°C      -14.50°C
  ECDH Derive                    -0.013°C      -12.80°C
  RSA Sign                       -0.007°C       -7.30°C
  KYBER Encaps                   -0.007°C       -7.20°C
  ECDH Keygen                    -0.003°C       -3.40°C
  KYBER Keygen                   -0.003°C       -3.30°C
  ECDSA Sign                     -0.003°C       -3.00°C
  RSA Encrypt                    +0.002°C       +2.00°C
  MLDSA Verify                   +0.002°C       +2.10°C
  RSA Decrypt                    +0.006°C       +6.00°C
  MLDSA Sign                     +0.006°C       +6.10°C
  ECDSA Verify                   +0.007°C       +6.80°C
  ECDSA Keygen                   +0.007°C       +7.00°C
  RSA Verify                     +0.013°C      +12.80°C
  RSA Keygen                     +0.015°C      +15.00°C
  KYBER Decaps 

In [112]:
print(f"\n  Hottest (most heat generated):")
for _, row in df_thermal.tail(3).iterrows():
    print(f"    {row['Algorithm']:<28} {row['Cumulative ΔT (°C)']:+.2f}°C")


  Hottest (most heat generated):
    RSA Verify                   +12.80°C
    RSA Keygen                   +15.00°C
    KYBER Decaps                 +20.30°C


In [113]:
print(f"\n  Coolest (net cooling effect):")
for _, row in df_thermal.head(3).iterrows():
    print(f"    {row['Algorithm']:<28} {row['Cumulative ΔT (°C)']:+.2f}°C")


  Coolest (net cooling effect):
    MLDSA Keygen                 -14.50°C
    ECDH Derive                  -12.80°C
    RSA Sign                     -7.30°C


---

## **S4 — Inferential Statistics**

This section answers whether the observed latency differences between algorithms are statistically significant and practically meaningful. The analysis follows a strict decision pipeline:

```mermaid
graph LR
    %% Definición de Estilos (Basado en tu configuración)
    classDef darkNode fill:#1e1e1e,stroke:#555555,color:#ffffff,stroke-width:1px;
    classDef highlightNode fill:#c084fc,stroke:#ffffff,color:#191919,font-weight:bold;
    classDef resultNode fill:#34d399,stroke:#ffffff,color:#191919,font-weight:bold;

    %% Estructura del Diagrama
    START([Normality test: Shapiro-Wilk]) --> CHECK{All distributions<br/>non-normal?}
    
    CHECK -->|Yes| NP[Non-parametric tests]
    NP --> MG[Multi-group comparison:<br/>Kruskal-Wallis H]
    MG --> PC[Pairwise comparison:<br/>Mann-Whitney U]
    PC --> ES[Effect size:<br/>rank-biserial correlation r]
    ES --> CORR[Multiple comparisons correction:<br/>Bonferroni]

    %% Aplicación de Estilos
    class START,NP darkNode;
    class MG,PC highlightNode;
    class ES,CORR resultNode;

    %% Configuración de Flechas y Fondo
    linkStyle default stroke:#aaaaaa,stroke-width:2px;
```

**Why effect size matters:** with n=1,000, Mann-Whitney U returns p<0.001 for almost any non-trivial difference. A p-value only tells you the difference is unlikely under the null — it says nothing about magnitude. The rank-biserial r (0 = no separation, 1 = complete separation) quantifies the practical importance of each difference.

**Bonferroni correction:** performing k pairwise tests at α=0.05 inflates the family-wise error rate. Bonferroni adjusts to α_corrected = 0.05/k. Given our effect sizes (r ≈ 1.0) and n=1,000, all comparisons remain significant after correction — this is reported explicitly rather than ignored.

In [ ]:
from scipy import stats as scipy_stats

# ── Shapiro-Wilk table
print("Shapiro-Wilk normality test (α = 0.05)\n")
print(f"{'Operation':<30} {'W':>8}  {'p-value':>12}  {'Normal?':>8}")
print("-" * 65)

sw_results = []
for name, df in sorted(df_collection.items()):
    # Shapiro-Wilk is unreliable above n=5000; subsample if needed
    sample = df['time_ms'].sample(min(1000, len(df)), random_state=42)
    w_stat, p_val = shapiro(sample)
    is_normal = p_val >= 0.05
    sw_results.append({
        'operation': name,
        'W'        : w_stat,
        'p_value'  : p_val,
        'normal'   : is_normal,
    })
    flag = "✓ yes" if is_normal else "✗ no "
    print(f"  {name:<28} {w_stat:>8.4f}  {p_val:>12.2e}  {flag}")

df_sw = pd.DataFrame(sw_results)
n_normal     = df_sw['normal'].sum()
n_non_normal = (~df_sw['normal']).sum()
print(f"\nResult: {n_non_normal}/16 non-normal, {n_normal}/16 normal")
print("→ Non-parametric tests required for all comparisons.")

Shapiro-Wilk normality test (α = 0.05)

Operation                             W       p-value   Normal?
-----------------------------------------------------------------
  ecdh256_derive                 0.6314      8.54e-42  ✗ no 
  ecdh256_keygen                 0.6378      1.54e-41  ✗ no 
  ecdsa256_keygen                0.6133      1.70e-42  ✗ no 
  ecdsa256_sign                  0.7612      1.04e-35  ✗ no 
  ecdsa256_verify                0.7287      1.91e-37  ✗ no 
  kyber512_decaps                0.3726      6.27e-50  ✗ no 
  kyber512_encaps                0.5405      4.47e-45  ✗ no 
  kyber512_keygen                0.8117      1.37e-32  ✗ no 
  mldsa44_keygen                 0.7686      2.73e-35  ✗ no 
  mldsa44_sign                   0.7998      2.22e-33  ✗ no 
  mldsa44_verify                 0.6156      2.08e-42  ✗ no 
  rsa2048_decrypt                0.2898      6.21e-52  ✗ no 
  rsa2048_encrypt                0.8214      6.46e-32  ✗ no 
  rsa2048_keygen                 0.87

In [62]:
def plot_qq(categories: dict) -> go.Figure:
    """
    5-panel Q-Q plot, one per algorithm family.
    Uses keygen for all families (most comparable operation type).
    """
    representatives = [
        ('kyber', 'keygen', 'Kyber-512 — KeyGen'),
        ('mldsa', 'sign',   'ML-DSA-44 — Sign'),
        ('rsa',   'sign',   'RSA-2048 — Sign'),
        ('ecdh',  'derive', 'ECDH-256 — Derive'),
        ('ecdsa', 'sign',   'ECDSA-256 — Sign'),
    ]

    fig = make_subplots(
        rows=1, cols=5,
        subplot_titles=[f'<b>{r[2]}</b>' for r in representatives],
        horizontal_spacing=0.05,
    )

    for col, (key, op, label) in enumerate(representatives, start=1):
        df  = categories[key][op]
        color = COLORS[key]
        data  = df['time_ms'].values

        # Theoretical quantiles from normal distribution
        (osm, osr), (slope, intercept, r_sq) = scipy_stats.probplot(data)
        # osm = theoretical quantiles, osr = ordered sample values

        # Scatter: observed vs theoretical
        fig.add_trace(go.Scatter(
            x    = osm,
            y    = osr,
            mode = 'markers',
            marker=dict(color=color, size=4, opacity=0.5,
                        line=dict(width=0)),
            showlegend=False,
            hovertemplate=(
                'Theoretical: %{x:.3f}<br>'
                f'Observed: %{{y:.4f}} ms<extra>{label}</extra>'
            ),
        ), row=1, col=col)

        # Reference line (what perfect normality looks like)
        x_line = np.array([osm.min(), osm.max()])
        y_line = slope * x_line + intercept
        fig.add_trace(go.Scatter(
            x    = x_line,
            y    = y_line,
            mode = 'lines',
            line = dict(color='white', width=1.2, dash='dash'),
            showlegend=False,
            hoverinfo='skip',
        ), row=1, col=col)

        # Annotate W statistic
        w_row = df_sw[df_sw['operation'].str.contains(f'{key}')].iloc[0]
        fig.add_annotation(
            x=0.05, y=0.95,
            xref=f'x{col} domain' if col > 1 else 'x domain',
            yref=f'y{col} domain' if col > 1 else 'y domain',
            text=f'W={w_row["W"]:.4f}',
            showarrow=False,
            font=dict(size=9, color=T['subtext']),
            align='left',
        )

        fig.update_xaxes(
            title_text='Theoretical quantiles' if col == 3 else '',
            showgrid=True,
            row=1, col=col,
        )
        fig.update_yaxes(
            title_text='Observed quantiles (ms)' if col == 1 else '',
            showgrid=True,
            row=1, col=col,
        )

    fig.update_layout(
        title=dict(
            text=(
                '<b>Q-Q plots — normality assessment per algorithm family</b><br>'
                '<sup>Points on diagonal = normal | Deviation = non-normal | '
                'Right-tail lift = positive skew</sup>'
            ),
            x=0.5,
        ),
        height=440,
        width=1200,
        margin=dict(t=100, b=60, l=70, r=40),
    )

    apply_theme(fig)
    return fig


fig_qq = plot_qq(category)
fig_qq.show()
save_fig(fig_qq, 'inf_qq_plots')

  Saved: figures/dark/inf_qq_plots.html
  Saved: figures/light/inf_qq_plots.html


In [63]:
def rank_biserial_r(u_stat: float, n1: int, n2: int) -> float:
    """
    Rank-biserial correlation: effect size for Mann-Whitney U.
    r = 1 - (2U) / (n1 * n2)
    Range: [-1, 1]. Absolute value interpreted as Cohen's conventions.
    """
    return 1 - (2 * u_stat) / (n1 * n2)

In [64]:
def effect_size_label(r: float) -> str:
    r = abs(r)
    if r >= 0.70: return 'Very large'
    if r >= 0.50: return 'Large'
    if r >= 0.30: return 'Medium'
    if r >= 0.10: return 'Small'
    return 'Negligible'

In [148]:
def run_inference(group_data: dict, group_label: str,
                  alpha: float = 0.05) -> pd.DataFrame:
    """
    Full inference pipeline for a set of algorithm groups.

    Args:
        group_data:  {'label': array_of_values, ...}
        group_label: 'KeyGen' | 'Sign' — for display
        alpha:       family-wise error rate (default 0.05)

    Returns:
        DataFrame with Mann-Whitney U results, effect sizes,
        and Bonferroni-corrected significance.
    """
    labels = list(group_data.keys())
    arrays = list(group_data.values())
    k      = len(arrays)

    # ── Kruskal-Wallis (multi-group)
    h_stat, p_kw = kruskal(*arrays)
    print(f"\n{'='*60}")
    print(f"  {group_label} — Kruskal-Wallis H test")
    print(f"{'='*60}")
    print(f"  H = {h_stat:.4f}  |  p = {p_kw:.2e}")
    conclusion = ("✓ Significant differences exist (p < 0.05)"
                  if p_kw < alpha
                  else "✗ No significant differences")
    print(f"  {conclusion}")
    print(f"  → Proceed to pairwise comparisons.\n")

    # ── Pairwise Mann-Whitney U + Effect size + Bonferroni
    from itertools import combinations
    pairs = list(combinations(range(k), 2))
    n_comparisons = len(pairs)
    alpha_bonf = alpha / n_comparisons

    print(f"  Pairwise Mann-Whitney U  (k={k}, comparisons={n_comparisons})")
    print(f"  Bonferroni α = {alpha:.2f}/{n_comparisons} = {alpha_bonf:.4f}\n")
    print(f"  {'Comparison':<35} {'U':>10}  {'p':>10}  {'r':>6}  "
          f"{'Effect':>12}  {'Bonf.':>6}")
    print(f"  {'-'*85}")

    rows = []
    for i, j in pairs:
        u_stat, p_val = mannwhitneyu(arrays[i], arrays[j],
                                     alternative='two-sided')
        r = rank_biserial_r(u_stat, len(arrays[i]), len(arrays[j]))
        sig_raw  = p_val < alpha
        sig_bonf = p_val < alpha_bonf
        label    = f'{labels[i]} vs {labels[j]}'
        effect   = effect_size_label(r)

        rows.append({
            'Comparison'    : label,
            'U'             : u_stat,
            'p-value'       : p_val,
            'r (effect)'    : round(r, 4),
            'Effect size'   : effect,
            'Sig (α=0.05)'  : '✓' if sig_raw  else '✗',
            'Sig (Bonf.)'   : '✓' if sig_bonf else '✗',
        })

        bonf_mark = '✓' if sig_bonf else '✗'
        print(f"  {label:<35} {u_stat:>10.0f}  {p_val:>10.2e}  "
            f"{r:>6.3f}  {effect:>12}  {bonf_mark:>6}")

    return pd.DataFrame(rows)

### **S4.2 — Kruskal-Wallis and Mann-Whitney U tests**

Running the full inference pipeline for KeyGen and Sign operation groups. Each call to `run_inference()` performs: Kruskal-Wallis (multi-group), pairwise Mann-Whitney U, rank-biserial r effect size, and Bonferroni correction.

In [149]:
# ── KeyGen comparison
keygen_groups = {
    'Kyber-512': category['kyber']['keygen']['time_ms'].values,
    'RSA-2048' : category['rsa']['keygen']['time_ms'].values,
    'ECDH-256' : category['ecdh']['keygen']['time_ms'].values,
}
df_inf_keygen = run_inference(keygen_groups, 'KeyGen')


  KeyGen — Kruskal-Wallis H test
  H = 2651.3413  |  p = 0.00e+00
  ✓ Significant differences exist (p < 0.05)
  → Proceed to pairwise comparisons.

  Pairwise Mann-Whitney U  (k=3, comparisons=3)
  Bonferroni α = 0.05/3 = 0.0167

  Comparison                                   U           p       r        Effect   Bonf.
  -------------------------------------------------------------------------------------
  Kyber-512 vs RSA-2048                        0    0.00e+00   1.000    Very large       ✓
  Kyber-512 vs ECDH-256                     5446    0.00e+00   0.989    Very large       ✓
  RSA-2048 vs ECDH-256                   1000000    0.00e+00  -1.000    Very large       ✓


In [71]:
# ── Sign comparison
sign_groups = {
    'ML-DSA-44': category['mldsa']['sign']['time_ms'].values,
    'RSA-2048' : category['rsa']['sign']['time_ms'].values,
    'ECDSA-256': category['ecdsa']['sign']['time_ms'].values,
}
df_inf_sign = run_inference(sign_groups, 'Sign')


  Sign — Kruskal-Wallis H test
  H = 2663.3016  |  p = 0.00e+00
  ✓ Significant differences exist (p < 0.05)
  → Proceed to pairwise comparisons.

  Pairwise Mann-Whitney U  (k=3, comparisons=3)
  Bonferroni α = 0.05/3 = 0.0167

  Comparison                                   U           p       r        Effect   Bonf.
  -------------------------------------------------------------------------------------
  ML-DSA-44 vs RSA-2048                      930    0.00e+00   0.998    Very large       ✓
  ML-DSA-44 vs ECDSA-256                 1000000    0.00e+00  -1.000    Very large       ✓
  RSA-2048 vs ECDSA-256                  1000000    0.00e+00  -1.000    Very large       ✓


In [73]:
print("\nFull results tables:")
display(df_inf_keygen.style.hide(axis='index')
        .set_caption('Pairwise comparisons — KeyGen'))
display(df_inf_sign.style.hide(axis='index')
        .set_caption('Pairwise comparisons — Sign'))


Full results tables:


Comparison,U,p-value,r (effect),Effect size,Sig (α=0.05),Sig (Bonf.)
Kyber-512 vs RSA-2048,0.000000,0.000000,1.000000,Very large,✓,✓
Kyber-512 vs ECDH-256,5445.500000,0.000000,0.989100,Very large,✓,✓
RSA-2048 vs ECDH-256,1000000.000000,0.000000,-1.000000,Very large,✓,✓


Comparison,U,p-value,r (effect),Effect size,Sig (α=0.05),Sig (Bonf.)
ML-DSA-44 vs RSA-2048,930.000000,0.000000,0.998100,Very large,✓,✓
ML-DSA-44 vs ECDSA-256,1000000.000000,0.000000,-1.000000,Very large,✓,✓
RSA-2048 vs ECDSA-256,1000000.000000,0.000000,-1.000000,Very large,✓,✓


### **S4.3 — Forest plots and significance heatmap**

Visual summary of all pairwise comparisons. The forest plot encodes three dimensions simultaneously: speedup (X axis), effect size r (marker size), and statistical significance (marker shape). The significance heatmap shows −log₁₀(p) for all 16×16 operation pairs.

In [ ]:
def plot_forest_v2(df_keygen: pd.DataFrame, df_sign: pd.DataFrame,
                   category: dict) -> go.Figure:

    med = {}
    for alg, ops in [
        ('kyber', ['keygen']),
        ('mldsa', ['sign']),
        ('rsa',   ['keygen', 'sign']),
        ('ecdh',  ['keygen']),
        ('ecdsa', ['sign']),
    ]:
        med[alg] = {op: float(category[alg][op]['time_ms'].median()) for op in ops}

    results = [
        {
            'label'  : 'Kyber-512 — KeyGen',
            'speedup': med['rsa']['keygen'] / med['kyber']['keygen'],
            'r'      : 1.00,
            'effect' : 'Very large',
            'color'  : COLORS['kyber'],
            'sig'    : True,
        },
        {
            'label'  : 'ECDH-256 — KeyGen',
            'speedup': med['rsa']['keygen'] / med['ecdh']['keygen'],
            'r'      : 1.00,
            'effect' : 'Very large',
            'color'  : COLORS['ecdh'],
            'sig'    : True,
        },
        {
            'label'  : 'ML-DSA-44 — Sign',
            'speedup': med['rsa']['sign'] / med['mldsa']['sign'],
            'r'      : 1.00,
            'effect' : 'Very large',
            'color'  : COLORS['mldsa'],
            'sig'    : True,
        },
        {
            'label'  : 'ECDSA-256 — Sign',
            'speedup': med['rsa']['sign'] / med['ecdsa']['sign'],
            'r'      : 1.00,
            'effect' : 'Very large',
            'color'  : COLORS['ecdsa'],
            'sig'    : True,
        },
    ]

    results.sort(key=lambda x: x['speedup'], reverse=True)

    max_sp  = max(r['speedup'] for r in results)
    # X range in log10: leave room on right for labels
    x_range = [np.log10(0.8), np.log10(max_sp) + 1.1]

    fig = go.Figure()

    # ── Background band to separate KeyGen / Sign groups
    keygen_labels = [r['label'] for r in results if 'KeyGen' in r['label']]
    sign_labels   = [r['label'] for r in results if 'Sign'   in r['label']]

    if keygen_labels:
        fig.add_hrect(
            y0=-0.5, y1=len(keygen_labels) - 0.5,
            fillcolor='rgba(255,255,255,0.03)',
            line_width=0,
        )
    if sign_labels:
        fig.add_hrect(
            y0=len(keygen_labels) - 0.5, y1=len(results) - 0.5,
            fillcolor='rgba(255,255,255,0.015)',
            line_width=0,
        )

    # ── Horizontal lines from 1× to marker
    for res in results:
        fig.add_shape(
            type='line',
            x0=1, x1=res['speedup'],
            y0=res['label'], y1=res['label'],
            xref='x', yref='y',
            line=dict(color=res['color'], width=2, dash='dot'),
        )

    # ── Markers
    for res in results:
        marker_size = 22 + res['r'] * 20   # 42px for r=1.0

        fig.add_trace(go.Scatter(
            x    = [res['speedup']],
            y    = [res['label']],
            mode = 'markers',
            name = res['label'],
            marker=dict(
                color   = res['color'],
                size    = marker_size,
                symbol  = 'diamond' if res['sig'] else 'circle-open',
                line    = dict(color='white', width=2),
                opacity = 0.92,
            ),
            showlegend=False,
            hovertemplate=(
                f"<b>{res['label']}</b><br>"
                f"Speedup: {res['speedup']:,.0f}×<br>"
                f"Effect size r: {res['r']:.2f} ({res['effect']})<br>"
                f"Bonferroni: {'✓' if res['sig'] else '✗'}"
                "<extra></extra>"
            ),
        ))

    # ── Labels: inside the plot area, right of marker
    # Use xshift in PIXELS — immune to log scale distortion
    for res in results:
        fig.add_annotation(
            x         = res['speedup'],
            y         = res['label'],
            text      = f"<b>{res['speedup']:,.0f}×</b>   r = {res['r']:.2f}   Effect: {res['effect']}",
            showarrow = False,
            xanchor   = 'left',
            xshift    = 26,          # pixel offset from marker edge
            yshift    = 0,
            font      = dict(size=12, color=T['text']),
        )

    # ── RSA baseline vertical line
    fig.add_vline(
        x=1,
        line=dict(color=COLORS['rsa'], width=2, dash='dash'),
    )
    fig.add_annotation(
        x=1, y=1.07, xref='x', yref='paper',
        text='RSA-2048<br>baseline (1×)',
        showarrow=False,
        font=dict(size=11, color=COLORS['rsa']),
        align='center',
        bgcolor=T['annotation_bg'],
        bordercolor=COLORS['rsa'],
        borderwidth=1,
        borderpad=4,
    )

    # ── Category labels (right margin)
    if keygen_labels:
        fig.add_annotation(
            xref='paper', yref='y',
            x=1.01, y=keygen_labels[len(keygen_labels)//2],
            text='<b>KeyGen</b>',
            showarrow=False,
            xanchor='left',
            font=dict(size=11, color=T['subtext']),
            textangle=-90,
        )
    if sign_labels:
        fig.add_annotation(
            xref='paper', yref='y',
            x=1.01, y=sign_labels[len(sign_labels)//2],
            text='<b>Sign</b>',
            showarrow=False,
            xanchor='left',
            font=dict(size=11, color=T['subtext']),
            textangle=-90,
        )

    # ── Bottom legend for marker shapes
    fig.add_annotation(
        xref='paper', yref='paper',
        x=0.5, y=-0.18,
        text=(
            '◆ Diamond = significant after Bonferroni correction   |   '
            'Marker size = effect size r   |   All comparisons: p < 0.001'
        ),
        showarrow=False,
        font=dict(size=10, color=T['subtext']),
        align='center',
    )

    fig.update_layout(
        title=dict(
            text=(
                '<b>Forest plot — speedups and effect sizes vs RSA-2048</b><br>'
                '<sup>Log scale | n=1,000 iterations per algorithm | '
                'Raspberry Pi 5 ARM64</sup>'
            ),
            x=0.5,
            font=dict(size=16),
        ),
        xaxis=dict(
            title    = 'Speedup vs RSA-2048 (× faster) — log scale',
            type     = 'log',
            showgrid = True,
            range    = x_range,
            tickvals = [1, 3, 10, 30, 100, 300, 1000, 3000, 10000],
            ticktext = ['1×', '3×', '10×', '30×', '100×', '300×', '1,000×',
                        '3,000×', '10,000×'],
            tickfont = dict(size=11),
        ),
        yaxis=dict(
            title     = '',
            autorange = 'reversed',
            tickfont  = dict(size=13),
        ),
        height  = 480,
        width   = 1150,
        margin  = dict(t=110, b=90, l=220, r=60),
        showlegend=False,
    )

    apply_theme(fig)
    return fig


fig_forest2 = plot_forest_v2(df_inf_keygen, df_inf_sign, category)
fig_forest2.show()
save_fig(fig_forest2, 'inf_forest_plot')

  Saved: figures/dark/inf_forest_plot.html
  Saved: figures/light/inf_forest_plot.html


In [92]:
def plot_significance_heatmap_v2(df_collection: dict) -> go.Figure:
    ops = sorted(df_collection.keys())
    n   = len(ops)

    # Clean short labels — no truncation
    def short_label(name: str) -> str:
        return (name
            .replace('kyber512_', 'Kyber-')
            .replace('mldsa44_',  'MLDSA-')
            .replace('rsa2048_',  'RSA-')
            .replace('ecdh256_',  'ECDH-')
            .replace('ecdsa256_', 'ECDSA-')
            .replace('keygen', 'KG')
            .replace('encaps', 'Enc')
            .replace('decaps', 'Dec')
            .replace('encrypt', 'Enc')
            .replace('decrypt', 'Dec')
            .replace('sign',   'Sign')
            .replace('verify', 'Vfy')
            .replace('derive', 'Drv')
        )

    short = [short_label(o) for o in ops]

    # Compute matrix
    matrix   = np.full((n, n), np.nan)
    n_comp   = n * (n - 1) / 2
    alpha_bonf = 0.05 / n_comp

    for i in range(n):
        for j in range(i + 1, n):
            _, p  = mannwhitneyu(
                df_collection[ops[i]]['time_ms'],
                df_collection[ops[j]]['time_ms'],
                alternative='two-sided',
            )
            val = -np.log10(max(p, 1e-300))
            matrix[i, j] = val
            matrix[j, i] = val

    bonf_threshold = -np.log10(alpha_bonf)
    vmax = np.nanmax(matrix)

    fig = go.Figure(go.Heatmap(
        z          = matrix,
        x          = short,
        y          = short,
        colorscale = [
            [0.0,  T['panel']],
            [bonf_threshold / vmax * 0.5, '#7f1d1d'],
            [bonf_threshold / vmax,        '#ff5757'],
            [0.8,  '#ff5757'],
            [1.0,  '#ffcccc'],
        ],
        zmin       = 0,
        zmax       = vmax,
        colorbar   = dict(
            title      = dict(text='−log₁₀(p)', side='right',
                              font=dict(size=12, color=T['text'])),
            tickvals   = [0, round(bonf_threshold, 1), 50, 100,
                          round(vmax, 0)],
            ticktext   = ['0',
                          f'Bonf. α\n({bonf_threshold:.1f})',
                          '50', '100',
                          f'{vmax:.0f}'],
            tickfont   = dict(size=10, color=T['subtext']),
            thickness  = 18,
            len        = 0.85,
            x          = 1.02,
        ),
        hovertemplate=(
            '<b>%{y} vs %{x}</b><br>'
            '−log₁₀(p) = %{z:.1f}<br>'
            'p ≈ 10^(−%{z:.0f})<extra></extra>'
        ),
        showscale=True,
        xgap=1, ygap=1,     # thin grid lines between cells
    ))

    fig.update_layout(
        title=dict(
            text=(
                '<b>Pairwise significance heatmap — all operations</b><br>'
                f'<sup>Color = −log₁₀(p) | Darker = more significant | '
                f'Bonferroni threshold: −log₁₀(p) = {bonf_threshold:.1f}</sup>'
            ),
            x=0.5,
        ),
        xaxis=dict(
            tickangle  = -45,       # angled but not vertical — readable
            tickfont   = dict(size=10, color=T['subtext']),
            side       = 'bottom',
            showgrid   = False,
        ),
        yaxis=dict(
            tickfont   = dict(size=10, color=T['subtext']),
            autorange  = 'reversed',
            showgrid   = False,
        ),
        height  = 640,
        width   = 720,
        margin  = dict(t=110, b=130, l=110, r=110),
    )

    apply_theme(fig)
    return fig


fig_heatmap2 = plot_significance_heatmap_v2(df_collection)
fig_heatmap2.show()
save_fig(fig_heatmap2, 'inf_significance_heatmap')

  Saved: figures/dark/inf_significance_heatmap.html
  Saved: figures/light/inf_significance_heatmap.html


### **S4.4 — Within-algorithm Spearman correlations**

Spearman ρ between the four benchmark metrics (latency, memory, ΔTemp, absolute temp) computed separately for each algorithm family. 

**Key question:** within RSA-2048, are slower iterations also hotter? If yes, temperature tracks latency. Within Kyber-512, if the correlation is near zero, it confirms that Kyber's thermal neutrality is a structural property of lattice arithmetic — not a side-effect of being faster.

In [117]:
def compute_spearman_matrix(df: pd.DataFrame) -> pd.DataFrame:
    """Spearman correlation matrix for the four benchmark metrics."""
    cols = ['time_ms', 'memory_system_mb', 'temp_delta_c', 'temp_absolute_c']
    available = [c for c in cols if c in df.columns]
    return df[available].dropna().corr(method='spearman')

In [151]:
def plot_correlation_heatmaps(category: dict) -> go.Figure:
    representatives = [
        ('rsa',   'keygen', 'RSA-2048<br>KeyGen'),
        ('kyber', 'keygen', 'Kyber-512<br>KeyGen'),
        ('mldsa', 'sign',   'ML-DSA-44<br>Sign'),
        ('ecdh',  'derive', 'ECDH-256<br>Derive'),
        ('ecdsa', 'sign',   'ECDSA-256<br>Sign'),
    ]

    short_labels = {
        'time_ms'          : 'Latency',
        'memory_system_mb' : 'Memory',
        'temp_delta_c'     : 'ΔTemp',
        'temp_absolute_c'  : 'Abs. Temp',
    }

    fig = make_subplots(
        rows=1, cols=5,
        subplot_titles=[f'<b>{r[2]}</b>' for r in representatives],
        horizontal_spacing=0.07,   # was 0.04 — more breathing room
    )

    for col, (key, op, label) in enumerate(representatives, start=1):
        df   = category[key][op]
        corr = compute_spearman_matrix(df)
        cols = corr.columns.tolist()
        short = [short_labels.get(c, c) for c in cols]

        show_colorbar = (col == 5)

        fig.add_trace(go.Heatmap(
            z          = corr.values,
            x          = short,
            y          = short,
            colorscale = [
                [0.0,  '#38bdf8'],
                [0.5,  T['panel']],
                [1.0,  '#ff5757'],
            ],
            zmin=-1, zmax=1,
            showscale  = show_colorbar,
            colorbar   = dict(
                title     = dict(text='ρ', font=dict(size=13, color=T['text'])),
                tickvals  = [-1, -0.5, 0, 0.5, 1],
                ticktext  = ['-1', '-0.5', '0', '0.5', '1'],
                tickfont  = dict(size=11, color=T['subtext']),
                thickness = 16,
                len       = 0.75,
                x         = 1.02,
            ),
            text        = [[f'{v:.2f}' for v in row] for row in corr.values],
            texttemplate= '%{text}',
            textfont    = dict(size=11, color='white'),
            hovertemplate=(
                '<b>%{y} vs %{x}</b><br>'
                'ρ = %{z:.3f}<extra>' + label.replace('<br>', ' ') + '</extra>'
            ),
            xgap=3, ygap=3,
        ), row=1, col=col)

        fig.update_xaxes(
            tickfont  = dict(size=10, color=T['subtext']),
            tickangle = -35,
            showgrid  = False,
            row=1, col=col,
        )
        fig.update_yaxes(
            tickfont  = dict(size=10, color=T['subtext']),
            showgrid  = False,
            row=1, col=col,
        )

    fig.update_layout(
        title=dict(
            text=(
                '<b>Within-algorithm Spearman correlations</b><br>'
                '<sup>ρ: blue = negative | neutral = none | red = positive | '
                'n=1,000 per operation</sup>'
            ),
            x=0.5,
            font=dict(size=15),
        ),
        height = 460,    # was 380 — more vertical room for labels
        width  = 1500,   # was 1200 — key change: wider canvas
        margin = dict(t=110, b=80, l=90, r=90),
    )

    apply_theme(fig)
    return fig

In [ ]:
# ── Key pairwise correlations with p-values
def print_key_correlations(category: dict):
    pairs = [
        ('time_ms', 'temp_delta_c',     'Latency ↔ ΔTemp'),
        ('time_ms', 'memory_system_mb', 'Latency ↔ Memory'),
    ]

    algs = [
        ('rsa',   'keygen', 'RSA-2048 KeyGen'),
        ('kyber', 'keygen', 'Kyber-512 KeyGen'),
        ('mldsa', 'sign',   'ML-DSA-44 Sign'),
        ('ecdh',  'derive', 'ECDH-256 Derive'),
        ('ecdsa', 'sign',   'ECDSA-256 Sign'),
    ]

    for var1, var2, pair_label in pairs:
        print(f"\n{pair_label}")
        print(f"  {'Algorithm':<22} {'ρ':>6}  {'p-value':>10}  {'Strength':>16}  {'Sig.':>5}")
        print(f"  {'-'*65}")
        for key, op, alg_label in algs:
            df = category[key][op]
            if var1 not in df.columns or var2 not in df.columns:
                continue
            data = df[[var1, var2]].dropna()
            rho, p = spearmanr(data[var1], data[var2])

            if abs(rho) < 0.10:   strength = 'Negligible'
            elif abs(rho) < 0.30: strength = 'Small'
            elif abs(rho) < 0.50: strength = 'Moderate'
            elif abs(rho) < 0.70: strength = 'Large'
            else:                  strength = 'Very large'

            sig = '✓' if p < 0.05 else '✗'
            print(f"  {alg_label:<22} {rho:>6.3f}  {p:>10.3e}  {strength:>16}  {sig:>5}")

    print()
    print("  Interpretation:")
    print("  Low ρ(latency↔ΔTemp) for Kyber vs high ρ for RSA would confirm")
    print("  that Kyber's thermal profile is structural, not speed-driven.")


fig_corr_matrix = plot_correlation_heatmaps(category)
fig_corr_matrix.show()
save_fig(fig_corr_matrix, 'spearman_correlation_heatmaps')

print_key_correlations(category)

  Saved: figures/dark/spearman_correlation_heatmaps.html
  Saved: figures/light/spearman_correlation_heatmaps.html

Latency ↔ ΔTemp
  Algorithm                   ρ     p-value          Strength   Sig.
  -----------------------------------------------------------------
  RSA-2048 KeyGen         0.016   6.218e-01        Negligible      ✗
  Kyber-512 KeyGen       -0.016   6.074e-01        Negligible      ✗
  ML-DSA-44 Sign         -0.010   7.496e-01        Negligible      ✗
  ECDH-256 Derive         0.003   9.160e-01        Negligible      ✗
  ECDSA-256 Sign          0.014   6.520e-01        Negligible      ✗

Latency ↔ Memory
  Algorithm                   ρ     p-value          Strength   Sig.
  -----------------------------------------------------------------
  RSA-2048 KeyGen        -0.080   1.152e-02        Negligible      ✓
  Kyber-512 KeyGen       -0.033   3.019e-01        Negligible      ✗
  ML-DSA-44 Sign         -0.032   3.148e-01        Negligible      ✗
  ECDH-256 Derive       

In [ ]:
def plot_forest_complete(category: dict) -> go.Figure:
    """
    Full pairwise forest plot. Runs Mann-Whitney U + rank-biserial r
    internally — no dependency on external result DataFrames.
    """
    from itertools import combinations

    # All groups for KeyGen and Sign comparisons
    keygen_groups = {
        'Kyber-512' : (category['kyber']['keygen']['time_ms'].values, 'kyber'),
        'RSA-2048'  : (category['rsa']['keygen']['time_ms'].values,   'rsa'),
        'ECDH-256'  : (category['ecdh']['keygen']['time_ms'].values,  'ecdh'),
    }
    sign_groups = {
        'ML-DSA-44' : (category['mldsa']['sign']['time_ms'].values,   'mldsa'),
        'RSA-2048'  : (category['rsa']['sign']['time_ms'].values,     'rsa'),
        'ECDSA-256' : (category['ecdsa']['sign']['time_ms'].values,   'ecdsa'),
    }

    def compute_pairs(groups: dict, category: str) -> list:
        rows = []
        labels = list(groups.keys())
        k = len(labels)
        n_comp = k * (k - 1) / 2
        alpha_bonf = 0.05 / n_comp

        for (l1, l2) in combinations(labels, 2):
            arr1, key1 = groups[l1]
            arr2, key2 = groups[l2]

            u_stat, p_val = mannwhitneyu(arr1, arr2, alternative='two-sided')
            r = 1 - (2 * u_stat) / (len(arr1) * len(arr2))

            med1 = float(np.median(arr1))
            med2 = float(np.median(arr2))

            # Speedup: faster / slower (always > 1)
            if med1 < med2:
                faster_label, slower_label = l1, l2
                faster_key,   slower_key   = key1, key2
                speedup = med2 / med1
            else:
                faster_label, slower_label = l2, l1
                faster_key,   slower_key   = key2, key1
                speedup = med1 / med2

            rows.append({
                'label'      : f'{faster_label} vs {slower_label}',
                'faster_key' : faster_key,
                'slower_key' : slower_key,
                'speedup'    : speedup,
                'r'          : abs(r),
                'p_val'      : p_val,
                'sig_bonf'   : p_val < alpha_bonf,
                'category'   : category,
                'U'          : u_stat,
            })
        return rows

    all_rows = compute_pairs(keygen_groups, 'KeyGen') + \
               compute_pairs(sign_groups,   'Sign')

    # Sort: category first, then descending speedup
    all_rows.sort(key=lambda x: (x['category'], -x['speedup']))
    df = pd.DataFrame(all_rows)

    fig = go.Figure()

    # ── Horizontal reference lines (1× to speedup point)
    for _, row in df.iterrows():
        fig.add_shape(
            type='line',
            x0=1, x1=row['speedup'],
            y0=row['label'], y1=row['label'],
            xref='x', yref='y',
            line=dict(
                color=COLORS[row['faster_key']],
                width=1.2,
                dash='dot',
            ),
        )

    # ── One scatter trace per category for legend
    for cat, cat_label in [('KeyGen', 'KeyGen operations'),
                            ('Sign',  'Sign operations')]:
        sub = df[df['category'] == cat]

        # Marker shape encodes speedup magnitude
        symbols = []
        for sp in sub['speedup']:
            if sp > 100:   symbols.append('star')
            elif sp > 10:  symbols.append('diamond')
            else:          symbols.append('circle')

        # Marker size encodes effect size r
        sizes = [14 + row['r'] * 22 for _, row in sub.iterrows()]

        # Color = faster algorithm's family color
        colors_pt = [COLORS[row['faster_key']] for _, row in sub.iterrows()]

        fig.add_trace(go.Scatter(
            x    = sub['speedup'],
            y    = sub['label'],
            mode = 'markers',
            name = cat_label,
            marker=dict(
                color  = colors_pt,
                size   = sizes,
                symbol = symbols,
                line   = dict(color='white', width=1.5),
                opacity= 0.92,
            ),
            hovertemplate=(
                '<b>%{y}</b><br>'
                'Speedup: %{x:.1f}×<br>'
                'Effect size r: %{customdata[0]:.3f}<br>'
                'p-value: %{customdata[1]:.2e}<br>'
                'Bonferroni: %{customdata[2]}<extra></extra>'
            ),
            customdata=[(row['r'], row['p_val'],
                         '✓' if row['sig_bonf'] else '✗')
                        for _, row in sub.iterrows()],
        ))

    # ── Speedup labels (xshift = px offset, safe on log scale)
    for _, row in df.iterrows():
        fig.add_annotation(
            x        = row['speedup'],
            y        = row['label'],
            text     = f"<b>{row['speedup']:.0f}×</b>  r={row['r']:.2f}",
            showarrow= False,
            xanchor  = 'left',
            xshift   = 14,
            font     = dict(size=10, color=T['text']),
        )

    # ── Category separator
    keygen_labels = [r['label'] for r in all_rows if r['category'] == 'KeyGen']
    sign_labels   = [r['label'] for r in all_rows if r['category'] == 'Sign']

    if keygen_labels and sign_labels:
        # Add a blank visual divider between KeyGen and Sign rows
        fig.add_hrect(
            y0=keygen_labels[-1], y1=sign_labels[0],
            fillcolor=T['grid'], opacity=0.3,
            line_width=0,
        )

    # ── RSA baseline
    fig.add_vline(
        x=1,
        line=dict(color=COLORS['rsa'], width=2, dash='dot'),
    )
    fig.add_annotation(
        x=1, y=1.04, xref='x', yref='paper',
        text='1× baseline',
        showarrow=False,
        font=dict(size=10, color=COLORS['rsa']),
        align='center',
    )

    # ── Shape legend
    fig.add_annotation(
        xref='paper', yref='paper',
        x=0.98, y=0.04,
        text='★ speedup >100×   ◆ 10–100×   ● <10×<br>'
             'Marker size = effect size r   |   All p < 0.001',
        showarrow=False,
        xanchor='right',
        font=dict(size=10, color=T['subtext']),
        bgcolor=T['annotation_bg'],
        bordercolor=T['annotation_border'],
        borderwidth=1, borderpad=6,
        align='right',
    )

    # ── Algorithm color legend
    for key, label in [('kyber', 'Kyber-512'), ('mldsa', 'ML-DSA-44'),
                        ('rsa', 'RSA-2048'), ('ecdh', 'ECDH-256'),
                        ('ecdsa', 'ECDSA-256')]:
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='markers',
            marker=dict(color=COLORS[key], size=10, symbol='circle',
                        line=dict(color='white', width=1)),
            name=label, showlegend=True,
        ))

    all_speedups = df['speedup'].tolist()
    max_sp = max(all_speedups)

    fig.update_layout(
        title=dict(
            text=(
                '<b>Forest plot — all pairwise comparisons</b><br>'
                '<sup>Color = faster algorithm | Size = effect size r | '
                'Shape = speedup magnitude | All p < 0.001 (Bonferroni corrected)</sup>'
            ),
            x=0.5,
        ),
        xaxis=dict(
            title  = 'Speedup (× faster) — log scale',
            type   = 'log',
            range  = [0, np.log10(max_sp) + 0.35],
            showgrid = True,
            tickvals = [1, 3, 10, 30, 100, 300, 1000, 3000],
            ticktext = ['1×', '3×', '10×', '30×', '100×', '300×', '1,000×', '3,000×'],
        ),
        yaxis=dict(
            title     = '',
            autorange = 'reversed',
            tickfont  = dict(size=11),
        ),
        legend=dict(
            title     = dict(text='Algorithm family:', font=dict(size=11)),
            orientation = 'v',
            x=1.02, y=0.98,
            font      = dict(size=10),
        ),
        height  = 480,
        width   = 1050,
        margin  = dict(t=110, b=80, l=230, r=200),
    )

    apply_theme(fig)
    return fig


fig_forest_final = plot_forest_complete(category)
fig_forest_final.show()
save_fig(fig_forest_final, 'inf_forest_complete')

  Saved: figures/dark/inf_forest_complete.html
  Saved: figures/light/inf_forest_complete.html


---

## **S5 — Quantum Threat Analysis: Shor's Algorithm**

This section addresses **OE4**: evaluation of the quantum threat to the cryptographic algorithms benchmarked above.

We implement Shor's algorithm at two levels:

- **Level 1 — Correct quantum simulation (N=15, N=21):** a proper quantum circuit with a unitary oracle that evaluates `f(x) = a^x mod N` in superposition. Not a classical approximation — a genuine quantum modular multiplication via `UnitaryGate`.
- **Level 2 — Theoretical extrapolation (N=2048 bits):** classical simulation of Shor for RSA-2048 is physically impossible (2^4098 amplitudes required). We measure circuit resources at N=15/21 and scale using Gidney & Ekerå (2021).

**Why this implementation is correct:** the oracle uses `UnitaryGate` — an explicit 2^n_aux × 2^n_aux unitary matrix that maps |y⟩ → |y · a^(2^i) mod N⟩. This is mathematically equivalent to controlled modular multiplication in the full Shor circuit, without the O(n³) gate decomposition that makes large N intractable on classical simulators.

**Honest scope declaration:** the extrapolation to RSA-2048 is always theoretical. This is standard practice in PQC literature — we make it explicit rather than obscuring it.

In [78]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import QFT
from qiskit.circuit.library import UnitaryGate
from qiskit_aer import AerSimulator
from math import gcd, log2, ceil
from fractions import Fraction
import numpy as np

def build_mod_mult_unitary(a_power: int, N: int, n_aux: int) -> np.ndarray:
    """
    Build the unitary matrix for modular multiplication by a_power mod N.

    Maps |y⟩ → |y * a_power mod N⟩ for y in {0, ..., N-1}.
    Pads with identity for y in {N, ..., 2^n_aux - 1} to make it
    a valid unitary over the full 2^n_aux dimensional Hilbert space.

    Args:
        a_power: the multiplier (a^(2^i) mod N for controlled qubit i)
        N:       the number to factor
        n_aux:   number of work qubits (ceil(log2(N)))

    Returns:
        2^n_aux × 2^n_aux unitary numpy array (complex128)
    """
    dim = 2 ** n_aux
    U   = np.zeros((dim, dim), dtype=complex)

    for y in range(dim):
        if y < N:
            # Modular multiplication: |y⟩ → |y·a mod N⟩
            y_out = (y * a_power) % N
        else:
            # Out-of-range: identity (padding)
            y_out = y
        U[y_out, y] = 1.0

    return U


def build_shor_circuit(N: int, a: int, n_count: int) -> QuantumCircuit:
    """
    Build a correct Shor circuit for factoring N with base a.

    The oracle uses UnitaryGate — mathematically equivalent to
    controlled modular multiplication, without the O(n³) gate
    decomposition that makes large N intractable to simulate.

    Args:
        N:       number to factor (must be odd, not prime power)
        a:       base (coprime to N, chosen s.t. gcd(a,N)=1)
        n_count: counting register size (≥ 2*ceil(log2(N)))

    Returns:
        QuantumCircuit ready for simulation
    """
    n_aux = ceil(log2(N + 1))   # work register: enough bits for {0,...,N-1}

    qr_count = QuantumRegister(n_count, name='count')
    qr_aux   = QuantumRegister(n_aux,   name='aux')
    cr       = ClassicalRegister(n_count, name='c')
    qc       = QuantumCircuit(qr_count, qr_aux, cr)

    # Step 1: Superposition on counting register
    qc.h(qr_count)

    # Step 2: Initialize work register to |1⟩ (y=1 so y·a^x = a^x)
    qc.x(qr_aux[0])

    # Step 3: Controlled-U_f for each counting qubit
    # Each qubit i controls multiplication by a^(2^i) mod N
    for i in range(n_count):
        a_power = pow(a, 2**i, N)           # classical: a^(2^i) mod N
        U_mat   = build_mod_mult_unitary(a_power, N, n_aux)
        gate    = UnitaryGate(U_mat, label=f'U_{a}^{2**i}')
        c_gate  = gate.control(1)           # make it controlled on qubit i
        qc.append(c_gate, [qr_count[i]] + list(qr_aux))

    # Step 4: Inverse QFT on counting register
    iqft = QFT(n_count, inverse=True, do_swaps=True)
    qc.append(iqft, qr_count)

    # Step 5: Measure counting register
    qc.measure(qr_count, cr)

    return qc


def extract_period(counts: dict, n_count: int, N: int) -> int | None:
    """
    Extract period r from measurement outcomes using continued fractions.

    The most probable measurement outcome m satisfies:
        m / 2^n_count ≈ k / r   for some integer k
    Continued fractions finds r from this approximation.

    Returns r if found and valid, None otherwise.
    """
    total_shots = sum(counts.values())
    top_states  = sorted(counts.items(), key=lambda x: x[1], reverse=True)

    for state_str, count in top_states[:8]:    # check top 8 outcomes
        m = int(state_str, 2)
        if m == 0:
            continue

        # Continued fractions approximation of m / 2^n_count
        frac = Fraction(m, 2**n_count).limit_denominator(N)
        r    = frac.denominator

        # Validate: r must be even and a^r ≡ 1 (mod N)
        if r % 2 == 0 and pow(int(pow(a_global, r//2, N_global)), 2, N_global) != 1:
            return r
        if pow(a_global, r, N_global) == 1 and r > 1:
            return r

    return None


print("✓ Shor oracle functions defined")
print("  build_mod_mult_unitary() — correct unitary for modular multiplication")
print("  build_shor_circuit()     — full Shor circuit with UnitaryGate oracle")
print("  extract_period()         — continued fractions period extraction")

✓ Shor oracle functions defined
  build_mod_mult_unitary() — correct unitary for modular multiplication
  build_shor_circuit()     — full Shor circuit with UnitaryGate oracle
  extract_period()         — continued fractions period extraction


In [80]:
N_global = 15
a_global = 7
n_count  = 4
SHOTS    = 4096

print("=" * 58)
print("  SHOR SIMULATION — RSA-15 (N=15, p=3, q=5, a=7)")
print("=" * 58)
print(f"\n  Target: factor N={N_global} = 3 × 5")
print(f"  Base  : a={a_global}  (gcd({a_global},{N_global}) = {gcd(a_global, N_global)})")
print(f"  Period: r=4  (7^4 mod 15 = {pow(7,4,15)}  → must equal 1)")
print(f"  n_count qubits: {n_count}  (2^{n_count} = {2**n_count} ≥ 2×⌈log₂(15)⌉ = 8)")

# Build circuit
qc = build_shor_circuit(N_global, a_global, n_count)

# Circuit metrics (before decomposition)
print(f"\n  Circuit metrics (UnitaryGate level):")
print(f"    Total qubits  : {qc.num_qubits}")
print(f"    Circuit depth : {qc.depth()}")
print(f"    Gate count    : {qc.size()}")
print(f"    Operations    : {dict(qc.count_ops())}")

# Decompose to basis gates for real metrics
qc_decomp = qc.decompose(reps=3)
cx_count  = sum(1 for g in qc_decomp.data if g.operation.name == 'cx')
print(f"\n  Decomposed to basis gates:")
print(f"    Depth         : {qc_decomp.depth()}")
print(f"    Total gates   : {qc_decomp.size()}")
print(f"    CNOT count    : {cx_count}")
print(f"    Gate breakdown: {dict(qc_decomp.count_ops())}")

  SHOR SIMULATION — RSA-15 (N=15, p=3, q=5, a=7)

  Target: factor N=15 = 3 × 5
  Base  : a=7  (gcd(7,15) = 1)
  Period: r=4  (7^4 mod 15 = 1  → must equal 1)
  n_count qubits: 4  (2^4 = 16 ≥ 2×⌈log₂(15)⌉ = 8)

  Circuit metrics (UnitaryGate level):
    Total qubits  : 8
    Circuit depth : 7
    Gate count    : 14
    Operations    : {'h': 4, 'c-unitary': 4, 'measure': 4, 'x': 1, 'IQFT': 1}

  Decomposed to basis gates:
    Depth         : 927
    Total gates   : 1384
    CNOT count    : 490
    Gate breakdown: {'u': 872, 'cx': 490, 'p': 18, 'measure': 4}


In [81]:
# Simulate
sim  = AerSimulator()
job  = sim.run(qc_decomp, shots=SHOTS)
counts = job.result().get_counts()

print(f"\n  Simulation results ({SHOTS} shots):")
print(f"  {'State':<16} {'Decimal':>8}  {'Count':>6}  {'Prob (%)':>8}  {'m/2^n':>8}  {'r candidate':>12}")
print(f"  {'-'*70}")

top_states = sorted(counts.items(), key=lambda x: x[1], reverse=True)[:8]
for state_str, count in top_states:
    m     = int(state_str, 2)
    prob  = count / SHOTS * 100
    ratio = m / 2**n_count
    frac  = Fraction(m, 2**n_count).limit_denominator(N_global)
    r_cand = frac.denominator if m > 0 else '—'
    print(f"  |{state_str}⟩ = {m:>4}   {count:>6}   {prob:>7.1f}%   {ratio:>8.4f}   {str(r_cand):>12}")

# Period extraction
r = extract_period(counts, n_count, N_global)

print(f"\n  Period extraction (continued fractions):")
if r:
    print(f"    r = {r}")
    half = pow(a_global, r//2, N_global)
    f1   = gcd(half - 1, N_global)
    f2   = gcd(half + 1, N_global)
    print(f"    a^(r/2) mod N = {a_global}^{r//2} mod {N_global} = {half}")
    print(f"    gcd(a^(r/2)-1, N) = gcd({half-1}, {N_global}) = {f1}")
    print(f"    gcd(a^(r/2)+1, N) = gcd({half+1}, {N_global}) = {f2}")
    if {f1, f2} - {1, N_global}:
        print(f"\n  ✓ FACTORED: {N_global} = {f1} × {f2}")
    else:
        print(f"\n  ✗ Trivial factors — retry with different a")
else:
    print("    Period not found in top outcomes — increase shots or n_count")


  Simulation results (4096 shots):
  State             Decimal   Count  Prob (%)     m/2^n   r candidate
  ----------------------------------------------------------------------
  |1000⟩ =    8     1040      25.4%     0.5000              2
  |0000⟩ =    0     1023      25.0%     0.0000              —
  |1100⟩ =   12     1017      24.8%     0.7500              4
  |0100⟩ =    4     1016      24.8%     0.2500              4

  Period extraction (continued fractions):
    r = 2
    a^(r/2) mod N = 7^1 mod 15 = 7
    gcd(a^(r/2)-1, N) = gcd(6, 15) = 3
    gcd(a^(r/2)+1, N) = gcd(8, 15) = 1

  ✓ FACTORED: 15 = 3 × 1


In [82]:
N_global = 21
a_global = 11
n_count  = 5
SHOTS    = 4096

print("=" * 58)
print("  SHOR SIMULATION — RSA-21 (N=21, p=3, q=7, a=11)")
print("=" * 58)
print(f"\n  Target: factor N={N_global} = 3 × 7")
print(f"  Base  : a={a_global}  (gcd({a_global},{N_global}) = {gcd(a_global, N_global)})")
print(f"  Period: r=6  (11^6 mod 21 = {pow(11,6,21)}  → must equal 1)")

qc21 = build_shor_circuit(N_global, a_global, n_count)
qc21_decomp = qc21.decompose(reps=3)
cx_count21  = sum(1 for g in qc21_decomp.data if g.operation.name == 'cx')

print(f"\n  Circuit metrics:")
print(f"    Total qubits  : {qc21.num_qubits}")
print(f"    Depth (decomp): {qc21_decomp.depth()}")
print(f"    Total gates   : {qc21_decomp.size()}")
print(f"    CNOT count    : {cx_count21}")

sim   = AerSimulator()
job   = sim.run(qc21_decomp, shots=SHOTS)
counts21 = job.result().get_counts()

print(f"\n  Top measurement outcomes ({SHOTS} shots):")
print(f"  {'State':<18} {'Dec':>5}  {'Count':>6}  {'Prob':>7}  {'r cand.':>10}")
print(f"  {'-'*55}")
for state_str, count in sorted(counts21.items(), key=lambda x: x[1], reverse=True)[:6]:
    m     = int(state_str, 2)
    prob  = count / SHOTS * 100
    frac  = Fraction(m, 2**n_count).limit_denominator(N_global) if m > 0 else None
    r_c   = frac.denominator if frac else '—'
    print(f"  |{state_str}⟩ = {m:>3}   {count:>6}   {prob:>6.1f}%   {str(r_c):>10}")

r21 = extract_period(counts21, n_count, N_global)
print(f"\n  Period extraction: r = {r21}")
if r21 and r21 % 2 == 0:
    half = pow(a_global, r21//2, N_global)
    f1   = gcd(half - 1, N_global)
    f2   = gcd(half + 1, N_global)
    if {f1, f2} - {1, N_global}:
        print(f"  ✓ FACTORED: {N_global} = {f1} × {f2}")

  SHOR SIMULATION — RSA-21 (N=21, p=3, q=7, a=11)

  Target: factor N=21 = 3 × 7
  Base  : a=11  (gcd(11,21) = 1)
  Period: r=6  (11^6 mod 21 = 1  → must equal 1)

  Circuit metrics:
    Total qubits  : 10
    Depth (decomp): 9906
    Total gates   : 14114
    CNOT count    : 5046

  Top measurement outcomes (4096 shots):
  State                Dec   Count     Prob     r cand.
  -------------------------------------------------------
  |10000⟩ =  16      724     17.7%            2
  |00000⟩ =   0      664     16.2%            —
  |10101⟩ =  21      488     11.9%           20
  |11011⟩ =  27      470     11.5%           19
  |00101⟩ =   5      470     11.5%           19
  |01011⟩ =  11      455     11.1%           20

  Period extraction: r = 2
  ✓ FACTORED: 21 = 1 × 3


In [84]:
def resource_scaling(n_bits: int, sim_n: int, sim_qubits: int,
                     sim_depth: int, sim_cnots: int) -> dict:
    """
    Scale circuit resources from simulated N to target n_bits RSA.

    Shor's algorithm complexity for n-bit RSA:
      Logical qubits : O(n)       — specifically ~2n + O(log n)
      Circuit depth  : O(n³)      — or O(n² log n) with optimizations
      CNOT count     : O(n³)

    We compute both naive O(n³) scaling and Gidney & Ekerå (2021)
    optimized estimates separately.
    """
    n_sim = ceil(log2(sim_n))       # bits in simulated N

    # Scaling factor from sim to target
    scale_qubits = n_bits / n_sim               # linear
    scale_depth  = (n_bits / n_sim) ** 3        # cubic (naive Shor)
    scale_cnots  = (n_bits / n_sim) ** 3

    return {
        'n_bits'              : n_bits,
        'logical_qubits_naive': int(sim_qubits * scale_qubits),
        'logical_qubits_gidney': 4 * n_bits + 2,            # Gidney 2021 formula
        'physical_qubits_100x': (4 * n_bits + 2) * 100,     # error correction ratio
        'physical_qubits_gidney2025': 400_000,               # Gidney optimized 2025
        'depth_naive'         : int(sim_depth * scale_depth),
        'cnots_naive'         : int(sim_cnots * scale_cnots),
        'classical_state_bits': n_bits * 4 + 2,              # qubits needed
    }


# Use RSA-15 metrics as base
metrics_15 = {
    'sim_n'    : 15,
    'sim_qubits': qc.num_qubits,
    'sim_depth' : qc_decomp.depth(),
    'sim_cnots' : cx_count,
}

res_2048 = resource_scaling(2048, **metrics_15)

In [85]:
print("=" * 62)
print("  THEORETICAL EXTRAPOLATION — RSA-2048")
print("=" * 62)
print()
print("  Source: Gidney & Ekerå (2021), DOI: 10.22331/q-2021-04-15-433")
print()
print(f"  {'Parameter':<40} {'Value':>18}")
print(f"  {'-'*60}")
print(f"  {'RSA key size (bits)':<40} {'2048':>18}")
print(f"  {'Logical qubits (naive 2n formula)':<40} {res_2048['logical_qubits_naive']:>18,}")
print(f"  {'Logical qubits (Gidney 2021: 4n+2)':<40} {res_2048['logical_qubits_gidney']:>18,}")
print(f"  {'Physical qubits (100:1 error corr.)':<40} {res_2048['physical_qubits_100x']:>18,}")
print(f"  {'Physical qubits (Gidney optimized 2025)':<40} {res_2048['physical_qubits_gidney2025']:>18,}")
print(f"  {'Circuit depth (naive O(n³) scaling)':<40} {res_2048['depth_naive']:>18,}")
print(f"  {'CNOT count (naive O(n³) scaling)':<40} {res_2048['cnots_naive']:>18,}")
print()
print(f"  Classical simulation feasibility:")
print(f"    State vector size: 2^{res_2048['classical_state_bits']} complex amplitudes")
print(f"    ≈ 10^{int(res_2048['classical_state_bits'] * 0.301):.0f} — physically impossible")
print()
print(f"  Q-Day estimate (cryptographically relevant QC):")
print(f"    Conservative: 2035–2040 (NIST, NSA)")
print(f"    Aggressive  : 2028–2032 (IBM, Google roadmaps)")
print(f"    Implication : migration window is now — not after Q-Day")


  THEORETICAL EXTRAPOLATION — RSA-2048

  Source: Gidney & Ekerå (2021), DOI: 10.22331/q-2021-04-15-433

  Parameter                                             Value
  ------------------------------------------------------------
  RSA key size (bits)                                    2048
  Logical qubits (naive 2n formula)                     4,096
  Logical qubits (Gidney 2021: 4n+2)                    8,194
  Physical qubits (100:1 error corr.)                 819,400
  Physical qubits (Gidney optimized 2025)             400,000
  Circuit depth (naive O(n³) scaling)         124,419,833,856
  CNOT count (naive O(n³) scaling)             65,766,686,720

  Classical simulation feasibility:
    State vector size: 2^8194 complex amplitudes
    ≈ 10^2466 — physically impossible

  Q-Day estimate (cryptographically relevant QC):
    Conservative: 2035–2040 (NIST, NSA)
    Aggressive  : 2028–2032 (IBM, Google roadmaps)
    Implication : migration window is now — not after Q-Day


In [86]:
# Summary comparison table
print()
print("=" * 62)
print("  RESOURCE COMPARISON: simulated vs RSA-2048 target")
print("=" * 62)
print(f"\n  {'Metric':<30} {'RSA-15 (sim)':>14} {'RSA-21 (sim)':>14} {'RSA-2048 (theor)':>18}")
print(f"  {'-'*78}")
print(f"  {'Total qubits':<30} {qc.num_qubits:>14} {qc21.num_qubits:>14} {'~4,098 logical':>18}")
print(f"  {'Circuit depth (decomp)':<30} {qc_decomp.depth():>14,} {qc21_decomp.depth():>14,} {'O(n³) — intractable':>18}")
print(f"  {'CNOT count':<30} {cx_count:>14,} {cx_count21:>14,} {'O(n³) — intractable':>18}")
print(f"  {'Classical simulation':<30} {'✓ feasible':>14} {'✓ feasible':>14} {'✗ impossible':>18}")
print(f"  {'Quantum resistance (Kyber)':<30} {'N/A':>14} {'N/A':>14} {'✓ LWE — secure':>18}")


  RESOURCE COMPARISON: simulated vs RSA-2048 target

  Metric                           RSA-15 (sim)   RSA-21 (sim)   RSA-2048 (theor)
  ------------------------------------------------------------------------------
  Total qubits                                8             10     ~4,098 logical
  Circuit depth (decomp)                    927          9,906 O(n³) — intractable
  CNOT count                                490          5,046 O(n³) — intractable
  Classical simulation               ✓ feasible     ✓ feasible       ✗ impossible
  Quantum resistance (Kyber)                N/A            N/A     ✓ LWE — secure


In [ ]:
# ── A) Measurement histogram
def plot_shor_histogram(counts: dict, n_count: int, N: int,
                        label: str, color: str) -> go.Figure:
    """
    Bar chart of measurement outcomes with period annotations.
    Peaks at multiples of 2^n_count / r are highlighted.
    """
    all_states = {format(i, f'0{n_count}b'): 0 for i in range(2**n_count)}
    all_states.update(counts)

    x_vals = [int(s, 2) for s in sorted(all_states.keys())]
    y_vals = [all_states[format(x, f'0{n_count}b')] for x in x_vals]
    total  = sum(y_vals)

    bar_colors = [color] * len(x_vals)

    fig = go.Figure(go.Bar(
        x            = x_vals,
        y            = [v/total*100 for v in y_vals],
        marker_color = bar_colors,
        opacity      = 0.8,
        hovertemplate='|%{x}⟩<br>Probability: %{y:.2f}%<extra></extra>',
    ))

    # Annotate expected peaks at multiples of 2^n_count / r
    r_expected = 4 if N == 15 else 6
    period_val = 2**n_count / r_expected
    for k in range(1, r_expected):
        peak_x = round(k * period_val)
        if 0 <= peak_x < 2**n_count:
            fig.add_vline(
                x=peak_x,
                line=dict(color='white', width=1.5, dash='dash'),
                annotation_text=f'k={k}<br>m={peak_x}',
                annotation_font=dict(size=9, color=T['subtext']),
                annotation_position='top',
            )

    fig.add_annotation(
        xref='paper', yref='paper', x=0.02, y=0.98,
        text=f'<b>{label}</b><br>Expected peaks at m = k·{2**n_count}/{r_expected}',
        showarrow=False, align='left',
        font=dict(size=10, color=T['text']),
        bgcolor=T['annotation_bg'],
        bordercolor=T['annotation_border'],
        borderwidth=1, borderpad=6,
    )

    fig.update_layout(
        title=dict(
            text=(
                f'<b>Shor measurement outcomes — {label}</b><br>'
                '<sup>Peaks at multiples of 2ⁿ/r → QFT period extraction</sup>'
            ),
            x=0.5,
        ),
        xaxis=dict(title='Measurement outcome (decimal)', showgrid=True, dtick=1),
        yaxis=dict(title='Probability (%)', showgrid=True),
        height=420, width=900,
        margin=dict(t=100, b=60, l=70, r=40),
        showlegend=False,
    )
    apply_theme(fig)
    return fig


fig_hist15 = plot_shor_histogram(counts,   n_count=4, N=15,
                                  label='RSA-15 (N=15, a=7)',
                                  color=COLORS['rsa'])
fig_hist15.show()
save_fig(fig_hist15, 'shor_histogram_rsa15')

fig_hist21 = plot_shor_histogram(counts21, n_count=5, N=21,
                                  label='RSA-21 (N=21, a=11)',
                                  color=COLORS['rsa'])
fig_hist21.show()
save_fig(fig_hist21, 'shor_histogram_rsa21')

  Saved: figures/dark/shor_histogram_rsa15.html
  Saved: figures/light/shor_histogram_rsa15.html


  Saved: figures/dark/shor_histogram_rsa21.html
  Saved: figures/light/shor_histogram_rsa21.html


In [ ]:
# ── B) Periodicity contrast: RSA vs Kyber LWE

def plot_periodicity_contrast() -> go.Figure:
    """
    Side-by-side: RSA periodic structure (exploitable by QFT)
    vs Kyber LWE noise (no structure → QFT cannot act).
    """
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            '<b>RSA: f(x) = 7ˣ mod 15</b><br>'
            '<sup>Periodic structure — exploitable by QFT</sup>',
            '<b>Kyber-512 (LWE): b = (As + e) mod q</b><br>'
            '<sup>Noise destroys periodicity — QFT cannot act</sup>',
        ),
        horizontal_spacing=0.12,
    )

    # RSA: periodic modular exponentiation
    x_rsa  = np.arange(0, 48)
    y_rsa  = np.array([pow(7, int(x), 15) for x in x_rsa]) / 15.0

    fig.add_trace(go.Scatter(
        x=x_rsa, y=y_rsa,
        mode='markers+lines',
        marker=dict(color=COLORS['rsa'], size=7,
                    line=dict(color='white', width=0.8)),
        line=dict(color=COLORS['rsa'], width=1.5),
        name='f(x) = 7ˣ mod 15',
        showlegend=True,
    ), row=1, col=1)

    # Period markers
    for xp in [4, 8, 12, 16, 20, 24, 28, 32, 36, 40, 44]:
        fig.add_vline(x=xp,
                      line=dict(color='rgba(255,255,255,0.15)', width=1, dash='dot'),
                      row=1, col=1)

    fig.add_annotation(
        x=24, y=1.12, xref='x', yref='y',
        text='<b>r = 4</b> (period)',
        showarrow=False,
        font=dict(size=12, color=COLORS['rsa']),
        bgcolor=T['annotation_bg'],
        bordercolor=COLORS['rsa'], borderwidth=1, borderpad=4,
        row=1, col=1,
    )

    # Kyber LWE: random-looking outputs
    np.random.seed(42)
    q       = 3329
    n_samp  = 150
    A_lwe   = np.random.randint(0, q, n_samp)
    s_lwe   = np.random.randint(-3, 4, n_samp)
    e_lwe   = np.random.normal(0, q * 0.05, n_samp).astype(int)
    b_lwe   = (A_lwe * s_lwe + e_lwe) % q
    b_norm  = b_lwe / q
    x_lwe   = np.arange(n_samp)

    # R² to show absence of structure
    coef    = np.polyfit(x_lwe, b_norm, 1)
    b_pred  = np.polyval(coef, x_lwe)
    r2      = 1 - np.sum((b_norm - b_pred)**2) / np.sum((b_norm - np.mean(b_norm))**2)

    fig.add_trace(go.Scatter(
        x=x_lwe, y=b_norm,
        mode='markers',
        marker=dict(
            color=b_norm,
            colorscale=[[0, COLORS['rsa']], [0.5, COLORS['kyber']], [1, '#ffffff']],
            size=5, opacity=0.7, line=dict(width=0),
        ),
        name='b = (As+e) mod q',
        showlegend=True,
    ), row=1, col=2)

    # Regression line (shows R²≈0)
    fig.add_trace(go.Scatter(
        x=x_lwe, y=b_pred,
        mode='lines',
        line=dict(color='rgba(255,255,255,0.3)', width=1.5, dash='dot'),
        name=f'Linear fit (R²={r2:.4f})',
        showlegend=True,
    ), row=1, col=2)

    fig.add_annotation(
        x=20, y=1.08, xref='x2', yref='y2',
        text=f'<b>R² = {r2:.4f} ≈ 0</b><br>No exploitable pattern',
        showarrow=False,
        font=dict(size=11, color=COLORS['kyber']),
        bgcolor=T['annotation_bg'],
        bordercolor=COLORS['kyber'], borderwidth=1, borderpad=4,
    )

    fig.update_xaxes(title_text='x (exponent)', showgrid=True, row=1, col=1)
    fig.update_xaxes(title_text='x (sample index)', showgrid=True, row=1, col=2)
    fig.update_yaxes(title_text='f(x) mod N [normalised 0–1]',
                     showgrid=True, range=[-0.05, 1.2], row=1, col=1)
    fig.update_yaxes(title_text='b mod q [normalised 0–1]',
                     showgrid=True, range=[-0.05, 1.2], row=1, col=2)

    fig.update_layout(
        height=480, width=1100,
        margin=dict(t=110, b=80, l=70, r=40),
        legend=dict(orientation='h', y=-0.2, x=0.5, xanchor='center'),
    )
    apply_theme(fig)
    return fig


fig_contrast = plot_periodicity_contrast()
fig_contrast.show()
save_fig(fig_contrast, 'shor_periodicity_contrast')

  Saved: figures/dark/shor_periodicity_contrast.html
  Saved: figures/light/shor_periodicity_contrast.html


In [ ]:
# ── C) Resource scaling vs n_bits

def plot_resource_scaling() -> go.Figure:
    """
    Log-log plot: logical qubits and CNOT count vs RSA key size.
    Shows measured points (N=15, N=21) and theoretical scaling.
    """
    n_bits_range = np.array([4, 5, 8, 16, 32, 64, 128, 256, 512, 1024, 2048])

    # Theoretical: 4n+2 logical qubits (Gidney)
    qubits_theory = 4 * n_bits_range + 2

    # Measured points
    measured = [
        (ceil(log2(15)), qc.num_qubits,   cx_count,   'N=15'),
        (ceil(log2(21)), qc21.num_qubits, cx_count21, 'N=21'),
    ]

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            '<b>Logical qubits vs key size</b><br>'
            '<sup>Linear scaling O(n) — Gidney 2021: 4n+2</sup>',
            '<b>CNOT count vs key size</b><br>'
            '<sup>Cubic scaling O(n³) — naive Shor</sup>',
        ),
        horizontal_spacing=0.12,
    )

    # Qubit scaling
    fig.add_trace(go.Scatter(
        x=n_bits_range, y=qubits_theory,
        mode='lines',
        line=dict(color=COLORS['rsa'], width=2),
        name='Theory: 4n+2 (Gidney 2021)',
    ), row=1, col=1)

    for n_sim, q_sim, _, lbl in measured:
        fig.add_trace(go.Scatter(
            x=[n_sim], y=[q_sim],
            mode='markers+text',
            marker=dict(color='white', size=12,
                        line=dict(color=COLORS['rsa'], width=2)),
            text=[lbl], textposition='top center',
            textfont=dict(size=10, color=T['subtext']),
            showlegend=(lbl == 'N=15'),
            name='Simulated',
        ), row=1, col=1)

    # RSA-2048 marker
    fig.add_trace(go.Scatter(
        x=[2048], y=[4*2048+2],
        mode='markers+text',
        marker=dict(color=COLORS['rsa'], size=14, symbol='star',
                    line=dict(color='white', width=1.5)),
        text=['RSA-2048<br>~8,194 qubits'], textposition='top left',
        textfont=dict(size=9, color=COLORS['rsa']),
        showlegend=False,
    ), row=1, col=1)

    # CNOT scaling
    base_n     = ceil(log2(15))
    base_cnots = cx_count
    cnots_cubic = base_cnots * (n_bits_range / base_n) ** 3

    fig.add_trace(go.Scatter(
        x=n_bits_range, y=cnots_cubic,
        mode='lines',
        line=dict(color=COLORS['rsa'], width=2),
        name='Theory: O(n³)',
        showlegend=True,
    ), row=1, col=2)

    for n_sim, _, c_sim, lbl in measured:
        fig.add_trace(go.Scatter(
            x=[n_sim], y=[c_sim],
            mode='markers',
            marker=dict(color='white', size=12,
                        line=dict(color=COLORS['rsa'], width=2)),
            showlegend=False,
        ), row=1, col=2)

    for col in [1, 2]:
        fig.update_xaxes(type='log', title_text='RSA key size (bits)',
                         showgrid=True, row=1, col=col)
    fig.update_yaxes(type='log', title_text='Logical qubits',
                     showgrid=True, row=1, col=1)
    fig.update_yaxes(type='log', title_text='CNOT count',
                     showgrid=True, row=1, col=2)

    fig.update_layout(
        height=460, width=1100,
        margin=dict(t=100, b=60, l=80, r=40),
        legend=dict(orientation='h', y=-0.15, x=0.5, xanchor='center'),
    )
    apply_theme(fig)
    return fig


fig_scaling = plot_resource_scaling()
fig_scaling.show()
save_fig(fig_scaling, 'shor_resource_scaling')

  Saved: figures/dark/shor_resource_scaling.html
  Saved: figures/light/shor_resource_scaling.html


### **S5.3 — Periodicity contrast: RSA vs Kyber LWE**

Visual demonstration of why QFT can attack RSA but not Kyber. RSA modular exponentiation f(x) = 7^x mod 15 has period r=4 — a structure the Quantum Fourier Transform can exploit. Kyber LWE output b = (As + e) mod q has R² ≈ 0 — statistically indistinguishable from uniform noise. QFT has nothing to latch onto.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# CONFIGURACIÓN VISUAL — estilo presentación oscura
# ============================================================
BG = "#191919"
PANEL = "#1e1e1e"
RSA_COLOR = "#ff5757"
KYBER_COLOR = "#ff914d"
GRID_COLOR = "#2a2a2a"
TEXT_COLOR = "#ffffff"
ACCENT = "#ffffff"

# ============================================================
# FIGURA 1: Función periódica RSA — f(x) = 7^x mod 15
# ============================================================
x_vals = np.arange(0, 16)
y_rsa = np.array([pow(7, int(x), 15) for x in x_vals])

# Extendemos para que se vea el patrón periódico claramente
x_ext = np.arange(0, 48)
y_ext = np.array([pow(7, int(x), 15) for x in x_ext])

fig_rsa = go.Figure()

# Área bajo la curva
fig_rsa.add_trace(go.Scatter(
    x=x_ext, y=y_ext,
    fill='tozeroy',
    fillcolor='rgba(255,68,68,0.08)',
    line=dict(color=RSA_COLOR, width=3),
    mode='lines+markers',
    marker=dict(size=8, color=RSA_COLOR,
                line=dict(color='white', width=1.5)),
    name='f(x) = 7ˣ mod 15'
))

# Anotaciones de período
for i, xpos in enumerate([4, 8, 12]):
    fig_rsa.add_vline(
        x=xpos,
        line=dict(color='rgba(255,255,255,0.25)', width=1.5, dash='dash')
    )

# Anotación período r=4
fig_rsa.add_annotation(
    x=2, y=15,
    text="r = 4",
    font=dict(size=22, color=RSA_COLOR, family="monospace"),
    showarrow=False,
    bgcolor="rgba(255,68,68,0.15)",
    bordercolor=RSA_COLOR,
    borderwidth=1,
    borderpad=6
)

fig_rsa.add_annotation(
    x=23, y=15.5,
    text="⚠ Periodicidad explotable por QFT",
    font=dict(size=16, color="#ffaa00"),
    showarrow=False,
)

fig_rsa.update_layout(
    title=dict(
        text="RSA: f(x) = 7ˣ mod 15",
        font=dict(size=28, color=ACCENT, family="monospace"),
        x=0.5
    ),
    paper_bgcolor=BG,
    plot_bgcolor=PANEL,
    font=dict(color=TEXT_COLOR, family="monospace"),
    xaxis=dict(
        title="x",
        gridcolor=GRID_COLOR,
        zerolinecolor=GRID_COLOR,
        tickfont=dict(size=14),
        title_font=dict(size=18)
    ),
    yaxis=dict(
        title="f(x)",
        gridcolor=GRID_COLOR,
        zerolinecolor=GRID_COLOR,
        tickfont=dict(size=14),
        title_font=dict(size=18)
    ),
    showlegend=False,
    width=900,
    height=550,
    margin=dict(l=60, r=40, t=80, b=60)
)



In [25]:


# ============================================================
# FIGURA 2: LWE — b = (As + e) mod q — caos sin patrón
# ============================================================
np.random.seed(42)
n_samples = 300
q = 3329  # módulo Kyber-512

# Generamos muestras LWE reales
A = np.random.randint(0, q, n_samples)
s = np.random.randint(0, 4, n_samples)  # secreto pequeño
e = np.random.normal(0, 3, n_samples).astype(int)  # error gaussiano
b = (A * s + e) % q

# Índices como eje x
x_lwe = np.arange(n_samples)

# Calculamos R² para mostrar
from numpy.polynomial import polynomial as P
coef = np.polyfit(x_lwe, b, 1)
b_pred = np.polyval(coef, x_lwe)
ss_res = np.sum((b - b_pred)**2)
ss_tot = np.sum((b - np.mean(b))**2)
r2 = 1 - ss_res/ss_tot

fig_lwe = go.Figure()

# Scatter de puntos LWE
fig_lwe.add_trace(go.Scatter(
    x=x_lwe, y=b,
    mode='markers',
    marker=dict(
        size=5,
        color=b,
        colorscale=[
    [0, "#ff5757"],
    [0.5, "#ff914d"],
    [1, "#ffffff"]
],
        opacity=0.7,
        line=dict(width=0)
    ),
    name='b = (As+e) mod q'
))

# Línea de regresión — muestra que R²≈0
fig_lwe.add_trace(go.Scatter(
    x=x_lwe, y=b_pred,
    mode='lines',
    line=dict(color='rgba(255,255,255,0.3)', width=2, dash='dot'),
    name=f'Regresión (R²={r2:.4f})'
))

fig_lwe.add_annotation(
    x=20, y=3700,
    text=f"R² ≈ {r2:.4f} ≈ 0",
    font=dict(size=22, color=KYBER_COLOR, family="monospace"),
    showarrow=False,
    bgcolor="rgba(255,145,77,0.12)",  # actualizado a tu naranja
    bordercolor=KYBER_COLOR,
    borderwidth=1,
    borderpad=6
)


fig_lwe.update_layout(
    title=dict(
        text="Kyber-512 (LWE): b = (As + e) mod q",
        font=dict(size=28, color=ACCENT, family="monospace"),
        x=0.5
    ),
    paper_bgcolor=BG,
    plot_bgcolor=PANEL,
    font=dict(color=TEXT_COLOR, family="monospace"),
    xaxis=dict(
        title="muestra i",
        gridcolor=GRID_COLOR,
        zerolinecolor=GRID_COLOR,
        tickfont=dict(size=14),
        title_font=dict(size=18)
    ),
    yaxis=dict(
        title="b mod q",
        gridcolor=GRID_COLOR,
        zerolinecolor=GRID_COLOR,
        tickfont=dict(size=14),
        title_font=dict(size=18)
    ),
    showlegend=False,
    width=900,
    height=550,
    margin=dict(l=60, r=40, t=80, b=60)
)




---


## **S6 — Summary: PQC vs Classical Cryptography on ARM64 IoT**



This section synthesises findings from §2–5 to answer the research question directly.

**Finding 1 — Latency:** PQC is not a performance penalty. Kyber-512 KeyGen is ~2,345× faster than RSA-2048. ML-DSA-44 Sign is ~6× faster. Compared to ECC (the current efficient baseline), full PQC adds only +0.14 ms per TLS handshake.

**Finding 2 — Thermal:** PQC algorithms run thermally neutral on ARM64. Kyber KeyGen produced a net negative cumulative ΔT over 1,000 iterations. Spearman ρ between latency and temperature is near zero across operations — the thermal advantage is structural, not a speed side-effect.

**Finding 3 — Quantum security:** RSA-2048 is theoretically breakable by Shor's algorithm (~8,194 logical qubits, Gidney & Ekerå 2021). Kyber-512 is based on LWE — no known quantum speedup exists. The R² ≈ 0 of LWE output (§5) demonstrates this empirically.

**Conclusion:** migration to PQC on ARM64 IoT hardware is **feasible today** — operationally, not just theoretically.

In [93]:
def build_master_table(category: dict) -> pd.DataFrame:
    """
    Unified comparison table across all metrics and dimensions.
    """
    # Representative operations per algorithm
    ops_map = {
        'Kyber-512':  ('kyber', 'keygen'),
        'ML-DSA-44':  ('mldsa', 'sign'),
        'RSA-2048':   ('rsa',   'keygen'),   # slowest = most conservative
        'ECDH-256':   ('ecdh',  'keygen'),
        'ECDSA-256':  ('ecdsa', 'sign'),
    }

    # Quantum resistance properties (literature-grounded)
    quantum_props = {
        'Kyber-512' : {'problem': 'LWE (lattice)', 'quantum_attack': 'None known',
                       'security_bits_classical': 128, 'security_bits_quantum': 128,
                       'nist_standard': 'FIPS 203 (2024)', 'resistant': True},
        'ML-DSA-44' : {'problem': 'MLWE/MSIS (lattice)', 'quantum_attack': 'None known',
                       'security_bits_classical': 128, 'security_bits_quantum': 128,
                       'nist_standard': 'FIPS 204 (2024)', 'resistant': True},
        'RSA-2048'  : {'problem': 'Integer factorisation', 'quantum_attack': "Shor O(n³)",
                       'security_bits_classical': 112, 'security_bits_quantum': 0,
                       'nist_standard': 'Legacy (deprecated 2030)', 'resistant': False},
        'ECDH-256'  : {'problem': 'Elliptic curve DLP', 'quantum_attack': "Shor O(n³)",
                       'security_bits_classical': 128, 'security_bits_quantum': 0,
                       'nist_standard': 'NIST P-256 (transitional)', 'resistant': False},
        'ECDSA-256' : {'problem': 'Elliptic curve DLP', 'quantum_attack': "Shor O(n³)",
                       'security_bits_classical': 128, 'security_bits_quantum': 0,
                       'nist_standard': 'NIST P-256 (transitional)', 'resistant': False},
    }

    rows = []
    for alg_label, (alg_key, op) in ops_map.items():
        df = category[alg_key][op]
        qp = quantum_props[alg_label]

        # All ops for this algorithm (thermal across all)
        all_dfs = [v for v in category[alg_key].values() if v is not None]
        temp_cumul_all = sum(float(d['temp_delta_c'].sum()) for d in all_dfs)

        rows.append({
            'Algorithm'           : alg_label,
            'Family'              : 'PQC' if qp['resistant'] else 'Classical',
            'Median latency (ms)' : float(df['time_ms'].median()),
            'P99 latency (ms)'    : float(df['time_ms'].quantile(0.99)),
            'CV (%)'              : float(df['time_ms'].std() / df['time_ms'].mean() * 100),
            'Cumul. ΔT (°C)'      : round(temp_cumul_all, 1),
            'Classical sec (bits)': qp['security_bits_classical'],
            'Quantum sec (bits)'  : qp['security_bits_quantum'],
            'Quantum attack'      : qp['quantum_attack'],
            'NIST status'         : qp['nist_standard'],
            'PQC ready'           : '✓' if qp['resistant'] else '✗',
        })

    df_master = pd.DataFrame(rows)

    # Style: color by family (PQC = orange/violet, Classical = red/blue/green)
    family_colors = {
        'Kyber-512' : COLORS['kyber'],
        'ML-DSA-44' : COLORS['mldsa'],
        'RSA-2048'  : COLORS['rsa'],
        'ECDH-256'  : COLORS['ecdh'],
        'ECDSA-256' : COLORS['ecdsa'],
    }

    def color_row(row):
        c = family_colors.get(row['Algorithm'], '#888')
        return [f'border-left: 4px solid {c}; padding-left: 8px'] + [''] * (len(row) - 1)

    display(
        df_master.drop(columns=['Family'])
        .style
        .apply(color_row, axis=1)
        .format({
            'Median latency (ms)': '{:.4f}',
            'P99 latency (ms)'   : '{:.4f}',
            'CV (%)'             : '{:.2f}',
            'Cumul. ΔT (°C)'     : '{:+.1f}',
        })
        .background_gradient(subset=['Median latency (ms)'], cmap='RdYlGn_r')
        .background_gradient(subset=['Quantum sec (bits)'],  cmap='RdYlGn')
        .hide(axis='index')
        .set_caption(
            'Master comparison table — latency, thermal, and quantum security '
            '(representative operation per algorithm)'
        )
    )

    return df_master


df_master = build_master_table(category)

Algorithm,Median latency (ms),P99 latency (ms),CV (%),Cumul. ΔT (°C),Classical sec (bits),Quantum sec (bits),Quantum attack,NIST status,PQC ready
Kyber-512,0.0847,0.1100,11.79,+9.8,128,128,None known,FIPS 203 (2024),✓
ML-DSA-44,0.6450,2.6344,60.32,-6.3,128,128,None known,FIPS 204 (2024),✓
RSA-2048,198.6064,717.6867,62.90,+28.5,112,0,Shor O(n³),Legacy (deprecated 2030),✗
ECDH-256,0.1179,0.1386,5.64,-16.2,128,0,Shor O(n³),NIST P-256 (transitional),✗
ECDSA-256,0.1852,0.2108,4.55,+10.8,128,0,Shor O(n³),NIST P-256 (transitional),✗


In [94]:
def normalise(values: list, invert: bool = False) -> list:
    """Min-max normalise. invert=True for metrics where lower=better."""
    mn, mx = min(values), max(values)
    if mx == mn:
        return [0.5] * len(values)
    normed = [(v - mn) / (mx - mn) for v in values]
    return [1 - v for v in normed] if invert else normed


def plot_radar(df_master: pd.DataFrame) -> go.Figure:
    categories_radar = [
        'Latency<br>score',
        'Consistency<br>(1/CV)',
        'Thermal<br>score',
        'Classical<br>security',
        'Quantum<br>security',
    ]

    algs = df_master['Algorithm'].tolist()

    # Raw values per axis
    raw = {
        'latency' : df_master['Median latency (ms)'].tolist(),
        'cv'      : df_master['CV (%)'].tolist(),
        'thermal' : df_master['Cumul. ΔT (°C)'].tolist(),
        'cl_sec'  : df_master['Classical sec (bits)'].tolist(),
        'qu_sec'  : df_master['Quantum sec (bits)'].tolist(),
    }

    # Normalise: latency invert (lower=better), cv invert, thermal invert
    scores = {
        'latency' : normalise(raw['latency'], invert=True),
        'cv'      : normalise(raw['cv'],      invert=True),
        'thermal' : normalise(raw['thermal'], invert=True),
        'cl_sec'  : normalise(raw['cl_sec'],  invert=False),
        'qu_sec'  : normalise(raw['qu_sec'],  invert=False),
    }

    family_colors = {
        'Kyber-512' : COLORS['kyber'],
        'ML-DSA-44' : COLORS['mldsa'],
        'RSA-2048'  : COLORS['rsa'],
        'ECDH-256'  : COLORS['ecdh'],
        'ECDSA-256' : COLORS['ecdsa'],
    }

    fig = go.Figure()

    for i, alg in enumerate(algs):
        r_vals = [
            scores['latency'][i],
            scores['cv'][i],
            scores['thermal'][i],
            scores['cl_sec'][i],
            scores['qu_sec'][i],
        ]
        # Close the polygon
        r_vals_closed = r_vals + [r_vals[0]]
        cats_closed   = categories_radar + [categories_radar[0]]

        color = family_colors[alg]
        # Hex to rgba for fill
        r_hex = int(color[1:3], 16)
        g_hex = int(color[3:5], 16)
        b_hex = int(color[5:7], 16)
        fill_color = f'rgba({r_hex},{g_hex},{b_hex},0.15)'

        fig.add_trace(go.Scatterpolar(
            r        = r_vals_closed,
            theta    = cats_closed,
            fill     = 'toself',
            fillcolor= fill_color,
            line     = dict(color=color, width=2.5),
            name     = alg,
            hovertemplate=(
                f'<b>{alg}</b><br>'
                '%{theta}: %{r:.2f}<extra></extra>'
            ),
        ))

    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                visible   = True,
                range     = [0, 1],
                tickvals  = [0, 0.25, 0.5, 0.75, 1.0],
                ticktext  = ['0', '0.25', '0.5', '0.75', '1.0'],
                tickfont  = dict(size=9, color=T['subtext']),
                gridcolor = T['grid'],
                linecolor = T['grid'],
            ),
            angularaxis=dict(
                tickfont  = dict(size=11, color=T['text']),
                linecolor = T['grid'],
                gridcolor = T['grid'],
            ),
            bgcolor = T['panel'],
        ),
        title=dict(
            text=(
                '<b>Multi-dimensional comparison — holistic IoT suitability</b><br>'
                '<sup>All axes normalised 0–1 (outer = better) | '
                'Quantum security: 1=PQC, 0=classically broken by Shor</sup>'
            ),
            x=0.5,
        ),
        legend=dict(
            orientation='v',
            x=1.12, y=0.5,
            font=dict(size=11, color=T['text']),
            bgcolor=T['annotation_bg'],
            bordercolor=T['annotation_border'],
            borderwidth=1,
        ),
        paper_bgcolor = T['bg'],
        font          = dict(color=T['text']),
        height=560, width=750,
        margin=dict(t=100, b=60, l=80, r=160),
    )

    return fig


fig_radar = plot_radar(df_master)
fig_radar.show()
save_fig(fig_radar, 'summary_radar')

  Saved: figures/dark/summary_radar.html
  Saved: figures/light/summary_radar.html


In [96]:
def plot_tls_scenarios(category: dict) -> go.Figure:
    med = {}
    for alg, ops in [
        ('kyber', ['keygen', 'encaps', 'decaps']),
        ('mldsa', ['keygen', 'sign', 'verify']),
        ('rsa',   ['keygen', 'encrypt', 'decrypt', 'sign', 'verify']),
        ('ecdh',  ['keygen', 'derive']),
        ('ecdsa', ['keygen', 'sign', 'verify']),
    ]:
        med[alg] = {
            op: float(category[alg][op]['time_ms'].median())
            for op in ops if category[alg].get(op) is not None
        }

    # TLS 1.3 handshake cost per scenario (ms)
    # Key exchange + authentication + verification
    scenarios = {
        'A: RSA-2048\n(legacy)': {
            'Key exchange'   : med['rsa']['encrypt'],
            'Authentication' : med['rsa']['sign'],
            'Verification'   : med['rsa']['verify'],
            'color'          : COLORS['rsa'],
        },
        'B: ECDH+ECDSA\n(current)': {
            'Key exchange'   : med['ecdh']['derive'],
            'Authentication' : med['ecdsa']['sign'],
            'Verification'   : med['ecdsa']['verify'],
            'color'          : COLORS['ecdh'],
        },
        'C: Kyber+MLDSA\n(full PQC)': {
            'Key exchange'   : med['kyber']['decaps'],
            'Authentication' : med['mldsa']['sign'],
            'Verification'   : med['mldsa']['verify'],
            'color'          : COLORS['kyber'],
        },
        'D: ECDH+MLDSA\n(hybrid)': {
            'Key exchange'   : med['ecdh']['derive'],
            'Authentication' : med['mldsa']['sign'],
            'Verification'   : med['mldsa']['verify'],
            'color'          : COLORS['mldsa'],
        },
    }

    op_colors = {
        'Key exchange'  : '#4878cf',
        'Authentication': '#6acc65',
        'Verification'  : '#d65f5f',
    }

    labels = list(scenarios.keys())
    ops    = ['Key exchange', 'Authentication', 'Verification']
    totals = {lbl: sum(v for k, v in d.items() if k != 'color')
              for lbl, d in scenarios.items()}

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            '<b>Per-handshake latency (ms)</b>',
            '<b>Cumulative cost — 1,000 handshakes/hour</b>',
        ),
        horizontal_spacing=0.14,
    )

    # Left: stacked bar per operation
    for op in ops:
        fig.add_trace(go.Bar(
            name         = op,
            x            = labels,
            y            = [scenarios[lbl][op] for lbl in labels],
            marker_color = op_colors[op],
            opacity      = 0.85,
            text         = [f"{scenarios[lbl][op]:.3f}" for lbl in labels],
            textposition = 'inside',
            textfont     = dict(size=9, color='white'),
            hovertemplate='<b>%{x}</b><br>' + op + ': %{y:.4f} ms<extra></extra>',
        ), row=1, col=1)

    # Total annotations
    for i, lbl in enumerate(labels):
        fig.add_annotation(
            x=lbl, y=totals[lbl] + 0.08,
            text=f'<b>{totals[lbl]:.3f} ms</b>',
            showarrow=False,
            font=dict(size=11, color=scenarios[lbl]['color']),
            row=1, col=1,
        )

    # Right: total cumulative (1000 handshakes = seconds)
    cumul_s = [totals[lbl] * 1000 / 1000 for lbl in labels]  # ms * 1000 -> s
    bar_colors = [scenarios[lbl]['color'] for lbl in labels]

    fig.add_trace(go.Bar(
        name         = 'Total time (s/hour)',
        x            = labels,
        y            = cumul_s,
        marker_color = bar_colors,
        opacity      = 0.85,
        text         = [f'{v:.3f} s' for v in cumul_s],
        textposition = 'outside',
        textfont     = dict(size=11, color=T['text']),
        showlegend   = False,
        hovertemplate='<b>%{x}</b><br>Total: %{y:.3f} s/hour<extra></extra>',
    ), row=1, col=2)

    # PQC baseline reference line
    pqc_total = totals['C: Kyber+MLDSA\n(full PQC)'] * 1000 / 1000
    fig.add_hline(
        y=pqc_total,
        line=dict(color=COLORS['kyber'], width=1.5, dash='dot'),
        annotation_text=f'PQC baseline ({pqc_total:.3f} s)',
        annotation_font_color=COLORS['kyber'],
        annotation_font_size=10,
        row=1, col=2,
    )

    fig.update_layout(
        title=dict(
            text=(
                '<b>TLS 1.3 handshake cost — IoT gateway scenarios</b><br>'
                '<sup>Raspberry Pi 5 ARM64 | 1,000 sensor connections/hour | '
                'Experimental data from E2</sup>'
            ),
            x=0.5,
        ),
        barmode  = 'stack',
        legend   = dict(
            title=dict(text='Operation:', font=dict(size=11)),
            orientation='h', y=-0.2, x=0.5, xanchor='center',
        ),
        height=540, width=1100,
        margin=dict(t=110, b=110, l=70, r=60),
    )

    fig.update_yaxes(title_text='Latency (ms)',          showgrid=True, row=1, col=1)
    fig.update_yaxes(title_text='Time (seconds/hour)',   showgrid=True, row=1, col=2)

    apply_theme(fig)
    return fig


fig_tls = plot_tls_scenarios(category)
fig_tls.show()
save_fig(fig_tls, 'summary_tls_scenarios')


  Saved: figures/dark/summary_tls_scenarios.html
  Saved: figures/light/summary_tls_scenarios.html
